# VQA-Med 2019 PA-SHE: Failure Ranking and Weight Sensitivity

This notebook compares Weighted Perturbation-Aware Semantic Hallucination
Entropy (Weighted PA-SHE) with Question-Aligned Semantic Nearest Neighbor
Entropy (QA-SNNE), introduced in
[Carlini et al., arXiv:2511.01458](https://arxiv.org/abs/2511.01458).

In [ ]:
!nvidia-smi

### Dependency environment

In [ ]:
# Dependencies are managed in ~/dissertation_2026/.venv before submission.
# Never install or upgrade packages inside an nbconvert job sharing that environment.
import sys
print("Using preconfigured Python environment:", sys.executable)

# Dataset: Complete Official VQA-Med 2019 Splits

Load the official **train**, **validation**, and **test** releases from the extracted Zenodo archives. Training uses only `train`; early stopping and uncertainty selection use only `validation`; final utility and failure-ranking evaluation use the held-out `test` split.

Set `VQAMED2019_ROOT` to the directory containing the extracted `ImageClef-2019-VQA-Med-Training`, `ImageClef-2019-VQA-Med-Validation`, and `VQAMed2019Test` folders. If the data are absent, the notebook downloads and verifies the three official Zenodo archives once, including extraction of the nested test-image archive. Set `VQAMED2019_AUTO_DOWNLOAD=0` to disable this behaviour. Official data: https://zenodo.org/records/10499039

The official release contains 12,792 training QA pairs over 3,200 images, 2,000 validation QA pairs over 500 images, and 500 manually reviewed test QA pairs over 500 images. It covers modality, plane, organ-system, and abnormality questions and is released under CC BY 4.0.


In [ ]:
import os
import re
import hashlib
import urllib.request
import zipfile
from pathlib import Path

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

VQAMED2019_EXPECTED_QA = {"train": 12792, "validation": 2000, "test": 500}
VQAMED2019_EXPECTED_IMAGES = {"train": 3200, "validation": 500, "test": 500}
VQAMED2019_TRAIN_MARKER = "All_QA_Pairs_train.txt"
VQAMED2019_AUTO_DOWNLOAD = os.environ.get(
    "VQAMED2019_AUTO_DOWNLOAD", "1"
).strip().lower() in {"1", "true", "yes"}
VQAMED2019_ARCHIVES = {
    "ImageClef-2019-VQA-Med-Training.zip": (
        "https://zenodo.org/records/10499039/files/ImageClef-2019-VQA-Med-Training.zip?download=1",
        "d53011394afa866a169a4fd86208a420",
    ),
    "ImageClef-2019-VQA-Med-Validation.zip": (
        "https://zenodo.org/records/10499039/files/ImageClef-2019-VQA-Med-Validation.zip?download=1",
        "ba990148074161d8206751e49c4acba3",
    ),
    "VQAMed2019Test.zip": (
        "https://zenodo.org/records/10499039/files/VQAMed2019Test.zip?download=1",
        "78dbfe55a86774b490cea64cc5f31f36",
    ),
}


def vqamed2019_md5(path, chunk_size=1024 * 1024):
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def vqamed2019_safe_extract(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = (destination / member.filename).resolve()
            if destination not in member_path.parents and member_path != destination:
                raise ValueError(f"Unsafe archive member: {member.filename}")
        archive.extractall(destination)


def vqamed2019_download_and_extract(target_root):
    target_root.mkdir(parents=True, exist_ok=True)
    for filename, (url, expected_md5) in VQAMED2019_ARCHIVES.items():
        archive_path = target_root / filename
        if archive_path.exists() and vqamed2019_md5(archive_path) != expected_md5:
            raise RuntimeError(
                f"Existing archive is incomplete or corrupted: {archive_path}. "
                "Remove that archive and rerun the notebook."
            )
        if not archive_path.exists():
            temporary_path = archive_path.with_suffix(archive_path.suffix + ".part")
            print(f"Downloading {filename} ...")
            urllib.request.urlretrieve(url, temporary_path)
            if vqamed2019_md5(temporary_path) != expected_md5:
                raise RuntimeError(
                    f"Checksum mismatch for {temporary_path}; download was not accepted."
                )
            temporary_path.replace(archive_path)
        print(f"Extracting {filename} ...")
        vqamed2019_safe_extract(archive_path, target_root)

    nested_test_archives = list(target_root.rglob("VQAMed2019_Test_Images.zip"))
    if not nested_test_archives:
        raise FileNotFoundError(
            "VQAMed2019_Test_Images.zip was not found after extracting VQAMed2019Test.zip"
        )
    for nested_archive in nested_test_archives:
        print(f"Extracting nested test images: {nested_archive}")
        vqamed2019_safe_extract(nested_archive, nested_archive.parent)


def vqamed2019_resolve_root():
    explicit = os.environ.get("VQAMED2019_ROOT")
    cwd = Path.cwd().resolve()
    project_home = Path.home() / "dissertation_2026"
    candidates = [
        Path(explicit).expanduser() if explicit else None,
        cwd / "data" / "VQA-Med-2019",
        cwd / "data",
        cwd / "VQA-Med-2019",
        cwd.parent / "data" / "VQA-Med-2019",
        project_home / "pa_she_vqamed_gpt2" / "data" / "VQA-Med-2019",
        project_home / "pa_she_vqamed_gpt2" / "data",
        project_home / "data" / "VQA-Med-2019",
    ]
    unique_candidates = []
    for candidate in candidates:
        if candidate is None:
            continue
        candidate = candidate.expanduser().resolve()
        if candidate not in unique_candidates:
            unique_candidates.append(candidate)
        if not candidate.is_dir():
            continue
        if (candidate / VQAMED2019_TRAIN_MARKER).is_file():
            return candidate
        if next(candidate.rglob(VQAMED2019_TRAIN_MARKER), None) is not None:
            return candidate
    attempted = "\n  - ".join(str(path) for path in unique_candidates)
    if VQAMED2019_AUTO_DOWNLOAD:
        target_root = (
            Path(explicit).expanduser().resolve()
            if explicit
            else (cwd / "data" / "VQA-Med-2019")
        )
        print("VQA-Med 2019 data not found; preparing official release at:", target_root)
        try:
            vqamed2019_download_and_extract(target_root)
        except Exception as error:
            raise RuntimeError(
                "Automatic VQA-Med 2019 setup failed. If CSF3 blocks downloads, "
                "download the three Zenodo archives on a login node and set "
                "VQAMED2019_ROOT. Original error: " + repr(error)
            ) from error
        if next(target_root.rglob(VQAMED2019_TRAIN_MARKER), None) is not None:
            return target_root
    raise FileNotFoundError(
        "VQA-Med 2019 data were not found. The loader looked under:\n  - "
        f"{attempted}\nDownload and extract the three official archives from "
        "https://zenodo.org/records/10499039, including the nested test-image "
        "archive. Then set, for example:\n"
        "export VQAMED2019_ROOT=\"$HOME/dissertation_2026/pa_she_vqamed_gpt2/data/VQA-Med-2019\""
    )


VQAMED2019_ROOT = vqamed2019_resolve_root()
print("Resolved VQA-Med 2019 root:", VQAMED2019_ROOT)


def vqamed2019_find(name, directory=False):
    direct = VQAMED2019_ROOT / name
    candidates = ([direct] if direct.exists() else []) + list(VQAMED2019_ROOT.rglob(name))
    candidates = sorted(
        {path.resolve() for path in candidates if path.is_dir() == directory},
        key=lambda path: (len(path.parts), str(path)),
    )
    if not candidates:
        kind = "directory" if directory else "file"
        raise FileNotFoundError(f"Required VQA-Med 2019 {kind} not found: {name}")
    return candidates[0]


def vqamed2019_reference_options(raw_answer):
    # Test references use # for alternatives and (...) for optional text.
    options = []
    for alternative in str(raw_answer).split("#"):
        alternative = re.sub(r"\s+", " ", alternative.strip())
        if not alternative:
            continue
        with_optional = re.sub(r"[()]", "", alternative)
        without_optional = re.sub(r"\([^)]*\)", "", alternative)
        for candidate in (with_optional, without_optional):
            candidate = re.sub(r"\s+", " ", candidate).strip()
            if candidate and candidate not in options:
                options.append(candidate)
    return options or [str(raw_answer).strip()]


def vqamed2019_image_index(image_dir):
    index = {}
    for path in sorted(image_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            if path.stem in index and index[path.stem] != path:
                raise ValueError(f"Duplicate image ID {path.stem} under {image_dir}")
            index[path.stem] = path
    if not index:
        raise FileNotFoundError(f"No extracted images found under {image_dir}")
    return index


def vqamed2019_load_split(annotation_path, image_dir, split_name):
    image_index = vqamed2019_image_index(image_dir)
    records = []
    for line_number, line in enumerate(
        annotation_path.read_text(encoding="utf-8-sig").splitlines(), start=1
    ):
        if not line.strip():
            continue
        fields = [field.strip() for field in line.split("|")]
        if len(fields) == 3:
            image_id, question, raw_answer = fields
            category = None
        elif len(fields) >= 4:
            image_id, category, question = fields[:3]
            raw_answer = "|".join(fields[3:]).strip()
        else:
            raise ValueError(
                f"Malformed {split_name} annotation line {line_number}: {line!r}"
            )
        image_id = Path(image_id).stem
        if image_id not in image_index:
            raise FileNotFoundError(
                f"Image {image_id!r} from {annotation_path.name}:{line_number} "
                f"was not found under {image_dir}"
            )
        references = vqamed2019_reference_options(raw_answer)
        records.append({
            "image_id": image_id,
            "image_path": str(image_index[image_id]),
            "category": category,
            "question": question,
            "answer": references[0],
            "answers": references,
        })
    expected = VQAMED2019_EXPECTED_QA[split_name]
    if len(records) != expected:
        raise ValueError(f"Expected {expected} {split_name} QA pairs; found {len(records)}")
    if len(image_index) != VQAMED2019_EXPECTED_IMAGES[split_name]:
        raise ValueError(
            f"Expected {VQAMED2019_EXPECTED_IMAGES[split_name]} {split_name} images; "
            f"found {len(image_index)}"
        )
    return records


train_annotation = vqamed2019_find("All_QA_Pairs_train.txt")
validation_annotation = vqamed2019_find("All_QA_Pairs_val.txt")
test_annotation = vqamed2019_find("VQAMed2019_Test_Questions_w_Ref_Answers.txt")
train_image_dir = vqamed2019_find("Train_images", directory=True)
validation_image_dir = vqamed2019_find("Val_images", directory=True)
test_image_dir = vqamed2019_find("VQAMed2019_Test_Images", directory=True)

vqamed2019_train = vqamed2019_load_split(train_annotation, train_image_dir, "train")
vqamed2019_validation = vqamed2019_load_split(
    validation_annotation, validation_image_dir, "validation"
)
vqamed2019_test = vqamed2019_load_split(test_annotation, test_image_dir, "test")
vqamed2019 = {
    "train": vqamed2019_train,
    "validation": vqamed2019_validation,
    "test": vqamed2019_test,
}

split_image_ids = {
    split: {record["image_id"] for record in records}
    for split, records in vqamed2019.items()
}
assert split_image_ids["train"].isdisjoint(split_image_ids["validation"])
assert split_image_ids["train"].isdisjoint(split_image_ids["test"])
assert split_image_ids["validation"].isdisjoint(split_image_ids["test"])

print({
    "data_root": str(VQAMED2019_ROOT),
    "train_QA": len(vqamed2019_train),
    "validation_QA": len(vqamed2019_validation),
    "test_QA": len(vqamed2019_test),
    "train_images": len(split_image_ids["train"]),
    "validation_images": len(split_image_ids["validation"]),
    "test_images": len(split_image_ids["test"]),
})
idx = 3
example = vqamed2019_test[idx]
with Image.open(example["image_path"]) as image:
    display_image = image.convert("RGB")
print(example.keys())
print("image resolution:", np.array(display_image).shape)
plt.figure(figsize=(5, 5))
plt.imshow(display_image)
plt.title(f"Q: {example['question']}\nA: {example['answer']}", fontsize=12)
plt.axis("off")
plt.show()


#Prepare Dataloader

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader


class VQAMed2019Dataset(Dataset):
    def __init__(self, records):
        self.dataset = records

        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        # Open lazily so the complete official split does not remain in RAM.
        with Image.open(sample["image_path"]) as image:
            img = self.transform(image.convert("RGB"))

        # --- Question & Answer ---
        question = str(sample["question"])
        answer = str(sample["answer"])

        return img, question, answer

    def references(self, idx):
        return list(self.dataset[idx].get("answers", [self.dataset[idx]["answer"]]))

# Use the complete official splits; do not resplit or subsample.
train_data = vqamed2019_train
val_data = vqamed2019_validation
test_data = vqamed2019_test

train_dataset = VQAMed2019Dataset(train_data)
val_dataset = VQAMed2019Dataset(val_data)
test_dataset = VQAMed2019Dataset(test_data)

print(
    f"Full official splits: train={len(train_dataset)}, "
    f"validation={len(val_dataset)}, test={len(test_dataset)}"
)


img, question, answer = train_dataset[1]
print("image resolution:", img.size())
plt.figure(figsize=(5, 5))
plt.axis("off")
plt.imshow(img.permute(1, 2, 0))
plt.title(f"Q: {question}\nA: {answer}", fontsize=12)
plt.show()


#Model Architecture

Paper: https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

GPT-2 uses a decoder-only transformer architecture with multiple model sizes; the commonly used GPT-2 Base model contains 12 transformer blocks (layers), a context window of 1024 tokens, a hidden embedding size of 768, and about 117 million parameters, while larger variants scale up to 48 transformer blocks and 1.5 billion parameters.

[1] Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. OpenAI blog, 1(8), 9.

###Cross-Attention Fusion

In [ ]:
import math
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model

####Cross-Attention Fusion###########
class CrossAttentionFusion(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=8, dropout=0.1):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(dropout)
        )

    def forward(self, text_embeds, image_embeds, text_att_mask=None):
        """
        text_embeds:  [B, T, 768]
        image_embeds: [B, N, 768]
        text attends to image
        """

        attended_text, attn_weights = self.cross_attn(
            query=text_embeds,
            key=image_embeds,
            value=image_embeds,
            need_weights=False
        )

        x = self.norm1(text_embeds + attended_text)
        x = self.norm2(x + self.ffn(x))

        return x

###Gated Cross-Attention Fusion

In [ ]:
#Write your code for Gated-Cross Attention Fusion

###MedVQA model (Mulitmodal GPT2)

In [ ]:
import math
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class MedVQA(nn.Module):
    def __init__(self, peft_config=None):
        super(MedVQA, self).__init__()

        # visual encoder
        model_name = "google/vit-base-patch16-224-in21k"
        self.visual_encoder = ViTModel.from_pretrained(model_name)

        # Freeze all parameters in visual encoder
        for param in self.visual_encoder.parameters():
            param.requires_grad = False

        # tokenizer
        self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.tokenizer.pad_token = self.tokenizer.eos_token  # end of string

        # gpt2 decoder
        gpt = GPT2LMHeadModel.from_pretrained('gpt2')
        self.gpt = get_peft_model(gpt, peft_config)
        # self.gpt.print_trainable_parameters()  # Verify trainable LoRA parameters
        self.fusion = CrossAttentionFusion(
            hidden_dim=768,
            num_heads=4
        )

    def forward(self, image, qa_inputs_ids, qa_att_mask):
        image_embeds = self.visual_encoder(image).last_hidden_state
        # [B, 197, 768]

        text_embeds = self.gpt.get_input_embeddings()(qa_inputs_ids)
        # [B, T, 768]

        fused_embeds = self.fusion(
            text_embeds=text_embeds,
            image_embeds=image_embeds,
            text_att_mask=qa_att_mask
        )
        # [B, T, 768]

        gpt_output = self.gpt(
            inputs_embeds=fused_embeds,
            attention_mask=qa_att_mask
        )
        return gpt_output.logits

# Model training or checkpoint reuse

This notebook reuses its own previously trained **VQA-Med 2019 PA-SHE** checkpoint by
default:

`checkpoints_pa_she_vqamed2019/best_model_ca_pa_she_vqamed2019.pth`

If the file is absent, training runs for up to **10 epochs** with validation
early stopping (patience **5**) and saves the best model there. Set
`REUSE_TRAINED_CHECKPOINT=0` to deliberately retrain and overwrite this
notebook's checkpoint. A PathVQA or VQA-RAD checkpoint must not be reused because
this dataset has different training examples and answer distributions.


In [ ]:
#Training Script for Multimodal GPT2 with LoRA
import os
import torch
import argparse
import torch.utils.data
import numpy as np
import random

from torch import nn
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer

import evaluate
from nltk.translate.bleu_score import corpus_bleu
from peft import  TaskType, LoraConfig

import warnings
warnings.filterwarnings('ignore')

REUSE_TRAINED_CHECKPOINT = os.environ.get(
    'REUSE_TRAINED_CHECKPOINT', '1'
).strip().lower() not in {'0', 'false', 'no'}


def load_vqa_checkpoint(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def adjust_learning_rate(optimizer, shrink_factor):
    print("\nDECAYING learning rate.")
    for param_group in optimizer.param_groups:
        param_group['lr'] = param_group['lr'] * shrink_factor
    print("The new learning rate is %f\n" % (optimizer.param_groups[0]['lr'],))

def train(args, train_dataloader, model, criterion, optimizer, epoch, tokenizer, device):
    model.train()
    total_loss = []

    for i, (images, questions, answers) in enumerate(train_dataloader, 0):
        # prepare prompts
        qa_prompt = [f'Question: {q}\nAnswer: {a}' for q, a in zip(questions, answers)]
        qa_prompt_inputs = tokenizer(qa_prompt, truncation=True, padding="max_length", max_length=int(args.seq_length), return_tensors="pt")

        # get labels
        labels = qa_prompt_inputs['input_ids'].clone()
        labels = labels.to(device)

        # for labels, mask question tokens and padding tokens
        for idx, q in enumerate(questions):
            q_prompt = f"Question: {q}\nAnswer: "
            q_length = len(tokenizer(q_prompt)["input_ids"]) - 1

            labels[idx, :q_length] = -100  # mask question
            eos_mask = (labels[idx] == tokenizer.eos_token_id)  # get all EOS position
            if eos_mask.sum() > 1:  # if more than 1 EOS
                first_eos_pos = eos_mask.nonzero()[0].item()  # get first EOS position
                labels[idx, (first_eos_pos+1):] = -100  # mask paddings, left one EOS

        # get logits and labels
        logits = model(
                image=images.to(device),
                qa_inputs_ids=qa_prompt_inputs['input_ids'].to(device),
                qa_att_mask=qa_prompt_inputs['attention_mask'].to(device)
        )

        # get shifted logits and labels
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        # compute loss
        shift_logits = shift_logits.view(-1, shift_logits.size(-1))
        shift_labels = shift_labels.view(-1)
        loss = criterion(shift_logits, shift_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss.append(loss.item())
        if i%50 == 0:
            print("Training - Epoch: {}/{}, Iteration: {}/{}, Training Loss: {:.6f}".format(epoch, args.epochs, i, len(train_dataloader), np.array(total_loss).mean()))


def validate(args, val_loader, model, criterion, epoch, tokenizer, device):
    total_loss = []
    model.eval()
    with torch.no_grad():
        for i, (images, questions, answers) in enumerate(val_loader, 0):
            # prepare prompts
            qa_prompt = [f'Question: {q}\nAnswer: {a}' for q, a in zip(questions, answers)]
            qa_prompt_inputs = tokenizer(qa_prompt, truncation=True, padding="max_length", max_length=int(args.seq_length), return_tensors="pt")

            # get labels
            labels = qa_prompt_inputs['input_ids'].clone()
            labels = labels.to(device)

            # for labels, mask question tokens and padding tokens
            answer_starts = []
            answer_ends = []
            for idx, q in enumerate(questions):
                q_prompt = f"Question: {q}\nAnswer: "
                q_length = len(tokenizer(q_prompt)["input_ids"]) - 1
                answer_starts.append(q_length+1)

                labels[idx, :q_length] = -100  # mask question
                eos_mask = (labels[idx] == tokenizer.eos_token_id)  # get all EOS position
                if eos_mask.sum() > 1:  # if more than 1 EOS
                    first_eos_pos = eos_mask.nonzero()[0].item()  # get first EOS position
                    labels[idx, (first_eos_pos+1):] = -100  # mask paddings, left one EOS
                    answer_ends.append(first_eos_pos)

            # get logits and labels
            logits = model(
                image=images.to(device),
                qa_inputs_ids=qa_prompt_inputs['input_ids'].to(device),
                qa_att_mask=qa_prompt_inputs['attention_mask'].to(device)
            )

            # get shifted logits and labels
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            # compute loss
            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)
            loss = criterion(shift_logits, shift_labels)
            total_loss.append(loss.item())

    return np.array(total_loss).mean()


def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)


def get_arg():
    parser = argparse.ArgumentParser(description='VisualQuestionAnswerGeneration')
    # Training parameters
    parser.add_argument('--epochs',         type=int,   default=10,   help='number of epochs to train for')
    parser.add_argument('--batch_size',     type=int,   default=64,   help='training and validation batch size')
    parser.add_argument('--workers',        type=int,   default=8,    help='for data-loading')
    parser.add_argument('--random_seed',    type=int,   default=42,   help='random seed')
    parser.add_argument('--seq_length',     type=int,   default=80,   help='sequence length for question and answer')
    parser.add_argument('--dropout', type=float, default=0.1, help='dropout')
    parser.add_argument('--early_stopping_patience', type=int, default=5,
                        help='stop after this many epochs without validation improvement')

    parser.add_argument('--dataset',        default='vqamed2019', help='dataset identifier')
    parser.add_argument('--lr',             type=float, default=0.0002,  help='0.0000001, 0.00000005')
    parser.add_argument('--checkpoint_dir', default='checkpoints_pa_she_vqamed2019/',
                        help='dataset-specific VQA-Med 2019 checkpoint path')

    args = parser.parse_args([])
    return args


if __name__ == '__main__':

    args = get_arg()
    seed_everything(args.random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f'Batch size: {args.batch_size}')
    print(f'Learning rate: {args.lr}')
    print(f'Random seed: {args.random_seed}')
    print(f'Sequence length: {args.seq_length}')
    print(f'Maximum epochs: {args.epochs}')
    print(f'Early-stopping patience: {args.early_stopping_patience}')

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    MODEL_CHECKPOINT_PATH = os.path.join(
        args.checkpoint_dir,
        'best_model_ca_pa_she_vqamed2019.pth',
    )
    reuse_checkpoint = (
        REUSE_TRAINED_CHECKPOINT
        and os.path.isfile(MODEL_CHECKPOINT_PATH)
    )
    checkpoint_saved_this_run = False
    start_epoch = 1
    epochs_since_improvement = 0
    best_val_loss = float('inf')

    print('Dataset: full official VQA-Med 2019 train/validation splits')
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=args.workers > 0,
    )
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=args.workers > 0,
    )

    print(
        'DataLoader configuration:',
        {
            'train_examples': len(train_dataset),
            'validation_examples': len(val_dataset),
            'batch_size': args.batch_size,
            'workers': args.workers,
        },
    )

    # init tokenizer and model
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"]
    )

    model = MedVQA(peft_config=lora_config)
    model = model.to(device)

    # for name, param in model.named_parameters():
    #     if param.requires_grad:
    #         print(name)

    pytorch_total_params = sum(p.numel() for p in model.parameters())
    print('model params: ', pytorch_total_params)

    # init optimizer and criterion
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    criterion = nn.CrossEntropyLoss(ignore_index=-100).to(device)

    # Reuse this notebook's validation-selected checkpoint unless retraining
    # is explicitly requested or the checkpoint does not exist.
    if reuse_checkpoint:
        print(
            'Reusing trained checkpoint; skipping epoch training:',
            MODEL_CHECKPOINT_PATH,
        )
        training_epochs = []
    else:
        if REUSE_TRAINED_CHECKPOINT:
            print(
                'No existing checkpoint found; training from scratch:',
                MODEL_CHECKPOINT_PATH,
            )
        else:
            print('Checkpoint reuse disabled; training from scratch.')
        print('Start training.')
        training_epochs = range(start_epoch, args.epochs + 1)

    for epoch in training_epochs:
        if epochs_since_improvement > 0 and epochs_since_improvement % 5 == 0:
            adjust_learning_rate(optimizer, 0.8)

        # train
        train(args, train_dataloader=train_dataloader, model=model, criterion=criterion, optimizer=optimizer,
              epoch=epoch, tokenizer=tokenizer, device=device)
        # validation
        val_loss = validate(args, val_loader=val_dataloader, model=model, criterion=criterion,
                            epoch=epoch, tokenizer=tokenizer, device=device)

        if val_loss < best_val_loss:  # save model with better validation loss
            epochs_since_improvement = 0
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_CHECKPOINT_PATH)
            checkpoint_saved_this_run = True
            model.tokenizer.save_pretrained(args.checkpoint_dir)
            print('Best validation loss, model saved.')
        else:
            epochs_since_improvement += 1
            print("\nEpochs since last improvement: %d\n" % (epochs_since_improvement,))

        if epochs_since_improvement >= args.early_stopping_patience:
            print(
                f'Early stopping at epoch {epoch}: validation loss did not improve '
                f'for {args.early_stopping_patience} consecutive epochs.'
            )
            break

    if not reuse_checkpoint:
        if not checkpoint_saved_this_run:
            raise RuntimeError(
                'Training finished without producing a validation checkpoint.'
            )
        print(f'End training. Best validation loss: {best_val_loss:.6f}')

    if not os.path.isfile(MODEL_CHECKPOINT_PATH):
        raise FileNotFoundError(
            f'Model checkpoint not found: {MODEL_CHECKPOINT_PATH}'
        )
    model.load_state_dict(
        load_vqa_checkpoint(MODEL_CHECKPOINT_PATH, device)
    )
    model.to(device)
    model.eval()
    print('Loaded validation-selected checkpoint:', MODEL_CHECKPOINT_PATH)


# Validation inference: qualitative sanity check

These examples are from validation. The official test split remains untouched
until the dataset-specific PA-SHE clustering has been selected and locked.


In [ ]:
# Validation-only prediction visualisations (test remains untouched)
from tqdm import tqdm
import evaluate

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model
from peft import  TaskType, LoraConfig
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

def greedy_search_single(image, question, model, tokenizer, max_length, device):
    model.eval()
    with torch.no_grad():
        # Prepare prompt and tokenize
        prompt_text = f"Question: {question}\nAnswer:"
        inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)
        input_ids = inputs['input_ids'].to(device)
        attention_mask = inputs['attention_mask'].to(device)

        # Pad to max_length
        padded_input_ids = torch.zeros((1, max_length), dtype=torch.long, device=device)
        padded_attention_mask = torch.zeros((1, max_length), dtype=torch.long, device=device)
        seq_len = input_ids.size(1)
        padded_input_ids[:, :seq_len] = input_ids
        padded_attention_mask[:, :seq_len] = attention_mask

        valid_length = seq_len
        generated_ids = []

        image = image.unsqueeze(0).to(device)  # Add batch dim

        for _ in range(max_length - seq_len):
            logits = model(
                image=image,
                qa_inputs_ids=padded_input_ids[:, :valid_length],
                qa_att_mask=padded_attention_mask[:, :valid_length]
            )

            last_logits = logits[0, valid_length - 1]  # shape: [vocab_size]
            next_token_id = torch.argmax(F.softmax(last_logits, dim=-1), dim=-1)

            if next_token_id.item() == tokenizer.eos_token_id:
                break

            padded_input_ids[0, valid_length] = next_token_id
            padded_attention_mask[0, valid_length] = 1
            valid_length += 1
            generated_ids.append(next_token_id.item())

        # Decode generated tokens
        answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        return answer


def inference_few_samples(sample_indices = [4, 5, 7]):
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"]
    )

    model = MedVQA(peft_config=lora_config)
    save_dir = MODEL_CHECKPOINT_PATH
    # save_dir = f'best_model_ca_lr1.pth'
    model.load_state_dict(load_vqa_checkpoint(save_dir, device))
    model.to(device)
    model.eval()

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_dataset = val_dataset
    print('Validation size (test remains untouched):', len(validation_dataset))

    # Create subplots
    fig, axes = plt.subplots(1, len(sample_indices), figsize=(12, 5))  # 1 row, 3 columns
    for ax, idx in zip(axes, sample_indices):
        img, question, answer_gt = validation_dataset[idx]

        # Run inference
        pred_answer = greedy_search_single(img, question, model, tokenizer, max_length=args.seq_length, device=device)
        img = img.permute(1, 2, 0)  # Convert from [C, H, W] to [H, W, C] for imshow
        ax.imshow(img)
        ax.set_title(f'Q: {question}\nA: {answer_gt}\nPred: {pred_answer}', fontsize=8)
        ax.axis('off')


inference_few_samples(sample_indices = [2, 80, 7])


# Validation inference: utility sanity check

This is not the final reported test utility. Official-test utility is computed
later from the greedy predictions generated after the clustering lock.


In [ ]:
# Validation-only utility sanity metrics (test remains untouched)
import os
import gc
import numpy as np
import random
import re

import torch
import torch.utils.data
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.transforms.functional import InterpolationMode
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model
from peft import  TaskType, LoraConfig

from PIL import Image
from tqdm import tqdm
import evaluate
rouge = evaluate.load("rouge")
import time
import math
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


def batch_greedy_search(images, questions, model, tokenizer, max_length, device):
    answers = []
    batch_size = len(questions)

    model.eval()
    with torch.no_grad():
        # Prepare the prompts for the entire batch
        prompt_texts = [f"Question: {q}\nAnswer:" for q in questions]

        # Tokenize the prompts with padding to handle varying lengths
        prompt_inputs = tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding='longest',
            add_special_tokens=False
        )

        # Prepare model inputs
        padded_input_ids = torch.zeros((batch_size, max_length), dtype=torch.long, device=device)
        padded_attention_mask = torch.zeros((batch_size, max_length), device=device)

        orig_length = prompt_inputs['input_ids'].size(1)
        padded_input_ids[:, :orig_length] = prompt_inputs['input_ids'].to(device)
        padded_attention_mask[:, :orig_length] = prompt_inputs['attention_mask'].to(device)

        images = images.to(device)

        # Initialize tensors to store generated tokens
        only_answer_ids = torch.empty((batch_size, 0), dtype=torch.long, device=device)

        # Track which sequences have finished generating
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Record each sample length (number of non-eos tokens)
        valid_lengths = padded_attention_mask.sum(dim=1).long()
        batch_indices = torch.arange(batch_size, device=device)

        for _ in range(max_length - orig_length):
            max_valid_lengths = valid_lengths.max().item()

            logits = model(
                image=images,
                qa_inputs_ids=padded_input_ids[:, :max_valid_lengths],
                qa_att_mask=padded_attention_mask[:, :max_valid_lengths]
            )

            last_valid_logits = logits[batch_indices, valid_lengths - 1, :]
            next_token_ids = torch.argmax(last_valid_logits, dim=-1)

            is_eos = next_token_ids == tokenizer.eos_token_id
            finished = finished | is_eos

            padded_input_ids[batch_indices, valid_lengths] = next_token_ids
            padded_attention_mask[batch_indices, valid_lengths] = 1
            valid_lengths += 1

            only_answer_ids = torch.cat(
                [only_answer_ids, next_token_ids.unsqueeze(1)],
                dim=1
            )

            if finished.all():
                break

        # Decode the generated tokens into strings
        generated_ids_cpu = only_answer_ids.cpu().tolist()  # Move to CPU and convert to list for processing
        for i in range(batch_size):
            # Find the first occurrence of eos_token_id to truncate the answer
            try:
                eos_index = generated_ids_cpu[i].index(tokenizer.eos_token_id)
                answer_ids = generated_ids_cpu[i][:eos_index]
            except ValueError:
                # If eos_token_id is not found, use all generated tokens
                answer_ids = generated_ids_cpu[i]

            # Decode the token IDs to a string, skipping special tokens
            answer = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
            answers.append(answer)

    return answers

def evaluate_vqa_split(args, data_loader, model, tokenizer, device):
    references = []
    hypotheses = []

    model.eval()
    with torch.no_grad():
        for i, (images, questions, answers) in enumerate(tqdm(data_loader), 0):
            generated_answers = batch_greedy_search(
                images,
                questions,
                model,
                tokenizer,
                max_length=args.seq_length,
                device=device
            )

            references.extend(answers)
            hypotheses.extend(generated_answers)

    return references, hypotheses

def normalize_vqa_metric_text(text):
    text = re.sub(r"[^a-z0-9%.\-\s]", " ", str(text).lower().strip())
    return re.sub(r"\s+", " ", text).strip() or "<empty>"


def get_nlp_mettics(references, hypotheses):
    references = [normalize_vqa_metric_text(text) for text in references]
    hypotheses = [normalize_vqa_metric_text(text) for text in hypotheses]
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    meteor = evaluate.load('meteor')

    # compute HF metrics
    results_bleu = bleu.compute(
        predictions=hypotheses, references=references, max_order=1
    )
    results_rouge = rouge.compute(predictions=hypotheses, references=references)
    results_meteor = meteor.compute(predictions=hypotheses, references=references)

    print("HuggingFace Metrics Results:")

    print(f"BLEU-1: {results_bleu['bleu']:.6f}")
    print(f"RougeL: {results_rouge['rougeL']:.6f}")
    print(f"Meteor: {results_meteor['meteor']:.6f}")


if __name__ == '__main__':
    # parameters
    random_seed = 42
    seed_everything(random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_dataset = val_dataset
    utility_eval_batch_size = int(os.environ.get("UTILITY_EVAL_BATCH_SIZE", "32"))
    validation_dataloader = DataLoader(
        validation_dataset, batch_size=utility_eval_batch_size, shuffle=False
    )
    print('Full validation size (test remains untouched):', len(validation_dataset))
    print('Utility evaluation batch size:', utility_eval_batch_size)

    # Reuse the validation-selected model loaded by the training/checkpoint cell.
    if "model" not in globals():
        raise RuntimeError("Run the training/checkpoint cell before utility evaluation.")
    # Training-only state is no longer needed and can retain substantial GPU memory.
    globals().pop("optimizer", None)
    globals().pop("criterion", None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model.to(device)
    model.eval()

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_references, validation_hypotheses = evaluate_vqa_split(
        args,
        data_loader=validation_dataloader,
        model=model,
        tokenizer=tokenizer,
        device=device,
    )
    get_nlp_mettics(validation_references, validation_hypotheses)


# Weighted Perturbation-Aware Semantic Hallucination Entropy (Weighted PA-SHE) for VQA-Med 2019

VQA-Med 2019 clustering is selected from a predeclared 13-candidate grid: Exact
text plus three thresholds each for SBERT, BGE, RoBERTa-NLI, and
DeBERTa-NLI on official validation at `ROUGE-L < 0.50`. Selection
uses the primary Equal condition weighting so that clustering choice is not
confounded with a test-set weight search. Overall and open-ended winners are
locked before official-test sampling. Equal remains the primary test weighting;
the other predeclared weights are sensitivity analyses.


## Experimental flow

```mermaid
flowchart TB
    A["Official VQA-Med 2019 validation"] --> B1["PA-SHE: four-condition sampled answers"]
    A --> B2["QA-SNNE: 20 original-input sampled answers"]
    B1 --> C1["Exact plus SBERT/BGE/RoBERTa/DeBERTa threshold grids"]
    C1 --> D1["Lock overall and open-ended PA-SHE clustering on validation"]
    B2 --> C2["ROUGE-L answer-similarity matrix"]
    B2 --> C3["Question-answer alignment:<br/>Embedding, NLI, or cross-encoder"]
    C2 --> D2["SNNE and bilaterally gated QA-SNNE"]
    C3 --> D2
    D2 --> E2["Select and lock QA-SNNE variant on validation"]
    F["Official VQA-Med 2019 test"] --> G["Apply locked configurations once"]
    D1 --> G
    E2 --> G
    G --> H["Compare Weighted PA-SHE, SNNE, QA-SNNE,<br/>semantic entropy, and token uncertainty"]
```

## 1. Configuration and reproducibility

The complete official validation split selects the dataset-specific VQA-Med 2019
clustering under Equal weighting. The official test split is sampled only
after that lock and is evaluated with all predeclared weight schemes. Sample
and feature caches are tied to the checkpoint and experimental configuration.


In [ ]:
# Install once if needed:
# !pip install -q pandas scipy scikit-learn sentence-transformers transformers rouge-score seaborn

import gc
import os
import hashlib
import json
from pathlib import Path
import math
import random
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TVF
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from rouge_score import rouge_scorer
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import average_precision_score, roc_auc_score
from tqdm.auto import tqdm

PASHE_MAX_EXAMPLES = None  # None = every example in each evaluated split
PASHE_NUM_SAMPLES = 10
PASHE_MAX_NEW_TOKENS = 24
PASHE_TEMPERATURE = 1.0
PASHE_TOP_P = 0.90
PASHE_RANDOM_SEED = 42
PASHE_PERTURBATION_VERSION = 2  # invalidates old samples after changing paraphrases/images


# Question-Aligned Semantic Nearest Neighbor Entropy (QA-SNNE).
# Paper: https://arxiv.org/abs/2511.01458
QA_SNNE_PAPER_ID = "2511.01458"
QA_SNNE_NUM_SAMPLES = int(os.environ.get("QA_SNNE_NUM_SAMPLES", "20"))
QA_SNNE_TEMPERATURE = 1.0
QA_SNNE_TOP_K = 50
QA_SNNE_TOP_P = 0.90
QA_SNNE_BETA = 10.0
QA_SNNE_TAU = 1.0
QA_SNNE_CACHE_SCHEMA_VERSION = 1
QA_SNNE_EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"
QA_SNNE_VARIANTS = {
    "Embedding": "qa_snne_embedding",
}
if QA_SNNE_NUM_SAMPLES < 2:
    raise ValueError("QA_SNNE_NUM_SAMPLES must be at least two.")

PASHE_LABEL_THRESHOLDS = [0.30, 0.50, 0.70]
PASHE_PRIMARY_LABEL_THRESHOLD = 0.50

# Predeclared method/threshold grid. Validation selects; test never does.
PASHE_SBERT_THRESHOLDS = [0.70, 0.80, 0.90]
PASHE_BGE_THRESHOLDS = [0.70, 0.80, 0.90]
PASHE_ROBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
PASHE_DEBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
# PA-SHE uses the sampled sequences' raw log-probabilities directly.
PASHE_SELECTION_CANDIDATES = (
    ["Exact text"]
    + [f"SBERT@{threshold:.2f}" for threshold in PASHE_SBERT_THRESHOLDS]
    + [f"BGE@{threshold:.2f}" for threshold in PASHE_BGE_THRESHOLDS]
    + [
        f"RoBERTa-NLI@{threshold:.2f}"
        for threshold in PASHE_ROBERTA_NLI_THRESHOLDS
    ]
    + [
        f"DeBERTa-NLI@{threshold:.2f}"
        for threshold in PASHE_DEBERTA_NLI_THRESHOLDS
    ]
)

PASHE_PRIMARY_WEIGHT_SCHEME = "Equal"
PASHE_CONDITION_WEIGHT_SCHEMES = {
    "Equal": {
        "original": 0.25,
        "weak": 0.25,
        "distorted": 0.25,
        "paraphrase": 0.25,
    },
    "Original-heavy": {
        "original": 0.55,
        "weak": 0.15,
        "distorted": 0.15,
        "paraphrase": 0.15,
    },
    "Moderate-original": {
        "original": 0.40,
        "weak": 0.20,
        "distorted": 0.20,
        "paraphrase": 0.20,
    },
    "Visual-heavy": {
        "original": 0.35,
        "weak": 0.25,
        "distorted": 0.25,
        "paraphrase": 0.15,
    },
    "Language-heavy": {
        "original": 0.35,
        "weak": 0.10,
        "distorted": 0.10,
        "paraphrase": 0.45,
    },
    "No paraphrase": {
        "original": 0.40,
        "weak": 0.30,
        "distorted": 0.30,
        "paraphrase": 0.00,
    },
    "No distortion": {
        "original": 0.50,
        "weak": 0.25,
        "distorted": 0.00,
        "paraphrase": 0.25,
    },
}

for scheme_name, scheme_weights in PASHE_CONDITION_WEIGHT_SCHEMES.items():
    if set(scheme_weights) != {"original", "weak", "distorted", "paraphrase"}:
        raise ValueError(f"{scheme_name} does not define every PASHE condition")
    if any(weight < 0 for weight in scheme_weights.values()):
        raise ValueError(f"{scheme_name} contains a negative weight")
    if not np.isclose(sum(scheme_weights.values()), 1.0):
        raise ValueError(f"{scheme_name} weights must sum to one")

PASHE_SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
PASHE_BGE_MODEL = "BAAI/bge-small-en-v1.5"
PASHE_ROBERTA_NLI_MODEL = "roberta-large-mnli"
PASHE_DEBERTA_NLI_MODEL = "microsoft/deberta-large-mnli"
PASHE_MODEL_BATCH_SIZE = 64
PASHE_CACHE_SCHEMA_VERSION = 4
PASHE_FEATURE_SCHEMA_VERSION = 7

required = [
    "model", "tokenizer", "device", "train_dataset", "val_dataset",
    "test_dataset", "MODEL_CHECKPOINT_PATH",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the original notebook through model loading/evaluation first. Missing: "
        + ", ".join(missing)
    )

PASHE_CHECKPOINT_PATH = Path(MODEL_CHECKPOINT_PATH).resolve()
if not PASHE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {PASHE_CHECKPOINT_PATH}")
PASHE_CACHE_DIR = (
    PASHE_CHECKPOINT_PATH.parent / "pa_she_cache_vqamed2019"
)
PASHE_CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 2. Perturbations and sampled generation

Four condition families are used: original, weak image, distorted image, and question paraphrase. Paraphrases use type-preserving templates and are accepted only when question type, negation, numbers, and laterality are unchanged. If no template matches, a validated image-context wrapper is used; a rejected rewrite remains unchanged. Radiology perturbations model bounded display/window and acquisition variation with gamma, contrast, blur, and noise. No crop, flip, or rotation is used, so anatomy, projection, and laterality are preserved.

In [ ]:
def pashe_normalize(text):
    return normalize_vqa_metric_text(text)


PASHE_QUESTION_ROUTER_VERSION = 1
PASHE_CLOSED_QUESTION_PREFIXES = (
    "is " , "are " , "was " , "were " , "do " , "does " ,
    "did " , "can " , "could " , "will " , "would " ,
    "has " , "have " , "had " , "should " , "may " , "might " ,
)


def pashe_predict_question_type(question):
    normalized = re.sub(r"\s+", " ", str(question).lower().strip())
    return (
        "Closed"
        if normalized.startswith(PASHE_CLOSED_QUESTION_PREFIXES)
        else "Open"
    )


PASHE_NEGATION_TERMS = {"no", "not", "without", "absent"}
PASHE_LATERALITY_TERMS = {"left", "right", "bilateral"}


def pashe_preserved_terms(text, vocabulary):
    tokens = set(re.findall(r"[a-z0-9]+", str(text).lower()))
    return tokens.intersection(vocabulary)


def pashe_paraphrase_constraints_hold(original, candidate):
    # Do not let the perturbation change the clinical question or its answer space.
    if not str(candidate).strip():
        return False
    if pashe_predict_question_type(original) != pashe_predict_question_type(candidate):
        return False
    original_numbers = re.findall(r"\b\d+(?:\.\d+)?\b", str(original))
    candidate_numbers = re.findall(r"\b\d+(?:\.\d+)?\b", str(candidate))
    if original_numbers != candidate_numbers:
        return False
    for protected in (PASHE_NEGATION_TERMS, PASHE_LATERALITY_TERMS):
        if pashe_preserved_terms(original, protected) != pashe_preserved_terms(candidate, protected):
            return False
    return pashe_normalize(original) != pashe_normalize(candidate)


def pashe_paraphrase(question):
    q = re.sub(r"\s+", " ", str(question).strip()).rstrip("?")
    if not q:
        return str(question)
    rules = [
        (r"^what kind of image is (?:this|shown)$", "Which imaging modality is shown?"),
        (r"^what modality (?:is used|does (?:this|the) image use)$", "Which imaging modality is shown?"),
        (r"^in which plane is (.+)$", r"What imaging plane is used for \1?"),
        (r"^which plane is (.+)$", r"What imaging plane is used for \1?"),
        (r"^which organ is (.+)$", r"What organ system is \1?"),
        (r"^what organ is (.+)$", r"Which organ system is \1?"),
        (r"^what is abnormal in (.+)$", r"Which abnormality is visible in \1?"),
        (r"^what is most alarming about (.+)$", r"Which abnormality is most evident in \1?"),
        (r"^what does (?:this|the) (?:image|picture) show$", "What is shown in the image?"),
        (r"^what is shown in (?:this|the) (?:image|picture)$", "What does the image show?"),
        (r"^is there (.+)$", r"Does the image show \1?"),
        (r"^does (?:this|the) image show (.+)$", r"Is \1 visible in the image?"),
        (r"^is (.+) present$", r"Does the image show \1?"),
        (r"^are there (.+)$", r"Does the image contain \1?"),
        (r"^can (.+) be seen$", r"Is \1 visible?"),
        (r"^where is (.+) located$", r"What is the location of \1?"),
        (r"^how many (.+) are (?:there|present)$", r"What number of \1 are present?"),
        (r"^what is (?:the )?diagnosis$", "Which diagnosis is most consistent with the image?"),
        (r"^what type of (.+) is (?:this|shown)$", r"Which type of \1 is shown?"),
        (r"^what is present$", "What finding is present?"),
    ]
    for pattern, replacement in rules:
        match = re.fullmatch(pattern, q, flags=re.IGNORECASE)
        if match:
            candidate = match.expand(replacement).strip()
            if pashe_paraphrase_constraints_hold(q, candidate):
                return candidate
    first_word = q.split(maxsplit=1)[0].lower()
    open_body = q[0].lower() + q[1:] if first_word in {"what", "where", "when", "why", "who", "which", "how"} else q
    fallback = (
        f"{q} according to the image?"
        if pashe_predict_question_type(q) == "Closed"
        else f"Based on the image, {open_body}?"
    )
    if pashe_paraphrase_constraints_hold(q, fallback):
        return fallback
    # Reject any unsafe rewrite instead of silently changing semantics.
    return f"{q}?"


def pashe_rng(seed):
    return random.Random(int(seed)) if seed is not None else random


def pashe_noise_like(image, standard_deviation, seed):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    return torch.randn(image.shape, generator=generator, dtype=image.dtype) * standard_deviation


def pashe_weak_image(image, seed=None):
    # Mild scanner/window variation; no anatomy-changing spatial transform.
    rng = pashe_rng(seed)
    x = image.detach().cpu().float().clamp(0, 1)
    x = TVF.adjust_gamma(x, rng.uniform(0.95, 1.05))
    x = TVF.adjust_contrast(x, rng.uniform(0.95, 1.05))
    sigma = rng.uniform(0.10, 0.35)
    x = TVF.gaussian_blur(x, [3, 3], [sigma, sigma])
    noise_seed = int(seed) + 1 if seed is not None else random.randrange(2**31)
    return (x + pashe_noise_like(x, rng.uniform(0.002, 0.008), noise_seed)).clamp(0, 1)


def pashe_distorted_image(image, seed=None):
    # Stronger but bounded radiology acquisition/display degradation.
    rng = pashe_rng(seed)
    x = image.detach().cpu().float().clamp(0, 1)
    x = TVF.adjust_gamma(x, rng.uniform(0.80, 1.20))
    x = TVF.adjust_contrast(x, rng.uniform(0.85, 1.15))
    sigma = rng.uniform(0.50, 1.25)
    x = TVF.gaussian_blur(x, [5, 5], [sigma, sigma])
    noise_seed = int(seed) + 1 if seed is not None else random.randrange(2**31)
    return (x + pashe_noise_like(x, rng.uniform(0.015, 0.040), noise_seed)).clamp(0, 1)


def pashe_top_p_filter(logits, top_p):
    if top_p >= 1.0:
        return logits
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
    remove = cumulative > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
    filtered = torch.full_like(logits, float("-inf"))
    return filtered.scatter(-1, sorted_indices, sorted_logits)


@torch.inference_mode()
def pashe_generate_one(image, question, do_sample=True):
    prompt = f"Question: {question}\nAnswer:"
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to(device)
    attention = encoded["attention_mask"].to(device)
    image_batch = image.unsqueeze(0).to(device)
    generated, logps = [], []

    for _ in range(PASHE_MAX_NEW_TOKENS):
        logits = model(
            image=image_batch,
            qa_inputs_ids=input_ids,
            qa_att_mask=attention,
        )[0, -1]
        scaled = logits / PASHE_TEMPERATURE
        log_probs = torch.log_softmax(scaled, dim=-1)
        if do_sample:
            filtered = pashe_top_p_filter(scaled, PASHE_TOP_P)
            next_id = torch.distributions.Categorical(logits=filtered).sample()
        else:
            next_id = torch.argmax(scaled)
        token_id = int(next_id.item())
        if token_id == tokenizer.eos_token_id:
            break
        generated.append(token_id)
        logps.append(float(log_probs[token_id].item()))
        input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
        attention = torch.cat(
            [attention, torch.ones((1, 1), dtype=attention.dtype, device=device)],
            dim=1,
        )

    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return {
        "answer": answer,
        "sequence_logprob": float(np.sum(logps)) if logps else -50.0,
    }



@torch.inference_mode()
def qa_snne_generate_samples(image, question, num_samples):
    """Generate all QA-SNNE samples in one autoregressive batch."""
    prompt = f"Question: {question}\nAnswer:"
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to(device).repeat(num_samples, 1)
    attention = encoded["attention_mask"].to(device).repeat(num_samples, 1)
    image_batch = image.unsqueeze(0).to(device).repeat(num_samples, 1, 1, 1)
    generated_ids = [[] for _ in range(num_samples)]
    finished = torch.zeros(num_samples, dtype=torch.bool, device=device)

    for _ in range(PASHE_MAX_NEW_TOKENS):
        next_logits = model(
            image=image_batch,
            qa_inputs_ids=input_ids,
            qa_att_mask=attention,
        )[:, -1, :] / QA_SNNE_TEMPERATURE

        top_k = min(QA_SNNE_TOP_K, next_logits.shape[-1])
        if top_k > 0:
            kth = torch.topk(next_logits, top_k, dim=-1).values[:, -1:]
            next_logits = next_logits.masked_fill(next_logits < kth, float("-inf"))
        next_logits = pashe_top_p_filter(next_logits, QA_SNNE_TOP_P)
        next_ids = torch.distributions.Categorical(logits=next_logits).sample()
        next_ids = torch.where(
            finished,
            torch.full_like(next_ids, tokenizer.eos_token_id),
            next_ids,
        )

        for sample_index, token_id in enumerate(next_ids.detach().cpu().tolist()):
            if not finished[sample_index] and token_id != tokenizer.eos_token_id:
                generated_ids[sample_index].append(token_id)
        finished = finished | (next_ids == tokenizer.eos_token_id)
        input_ids = torch.cat([input_ids, next_ids[:, None]], dim=1)
        attention = torch.cat([
            attention,
            torch.ones((num_samples, 1), dtype=attention.dtype, device=device),
        ], dim=1)
        if bool(finished.all()):
            break

    return [
        tokenizer.decode(token_ids, skip_special_tokens=True).strip()
        for token_ids in generated_ids
    ]

def pashe_collect_example(
    dataset,
    dataset_index,
    split_name,
):
    image, question, reference = dataset[dataset_index]
    references = dataset.references(dataset_index)
    paraphrase = pashe_paraphrase(question)
    split_offset = 0 if str(split_name).lower().startswith("val") else 10_000_000
    perturbation_seed = PASHE_RANDOM_SEED + split_offset + int(dataset_index) * 1000
    condition_inputs = {
        "original": [(image.clone(), question) for _ in range(PASHE_NUM_SAMPLES)],
        "weak": [
            (pashe_weak_image(image, perturbation_seed + 100 + sample_index), question)
            for sample_index in range(PASHE_NUM_SAMPLES)
        ],
        "distorted": [
            (pashe_distorted_image(image, perturbation_seed + 200 + sample_index), question)
            for sample_index in range(PASHE_NUM_SAMPLES)
        ],
        "paraphrase": [(image.clone(), paraphrase) for _ in range(PASHE_NUM_SAMPLES)],
    }
    records = []
    for condition, inputs in condition_inputs.items():
        for condition_image, condition_question in inputs:
            record = pashe_generate_one(condition_image, condition_question, do_sample=True)
            record["condition"] = condition
            records.append(record)
    greedy = pashe_generate_one(
        image,
        question,
        do_sample=False,
    )["answer"]
    return {
        "cache_schema_version": PASHE_CACHE_SCHEMA_VERSION,
        "split": str(split_name),
        "dataset_index": int(dataset_index),
        "question": question,
        "paraphrase": paraphrase,
        "paraphrase_changed": pashe_normalize(question) != pashe_normalize(paraphrase),
        "reference": str(reference),
        "references": [str(value) for value in references],
        "greedy": greedy,
        "records": records,
    }

## 3. Dataset-specific clustering backends

Validation compares Exact text with SBERT/BGE cosine thresholds `0.70`,
`0.80`, and `0.90`, plus mutual RoBERTa/DeBERTa-NLI thresholds `0.35`,
`0.50`, and `0.65`. Score matrices are reused across thresholds, and test
constructs only the validation-locked overall and open-ended clusterings.


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

pashe_sbert = SentenceTransformer(PASHE_SBERT_MODEL, device=str(device))
pashe_bge = SentenceTransformer(PASHE_BGE_MODEL, device=str(device))


def pashe_load_nli(model_name):
    tokenizer_nli = AutoTokenizer.from_pretrained(model_name)
    model_nli = AutoModelForSequenceClassification.from_pretrained(
        model_name
    ).to(device).eval()
    entailment_id = next(
        (
            int(index)
            for index, label in model_nli.config.id2label.items()
            if "entail" in str(label).lower()
        ),
        2,
    )
    return tokenizer_nli, model_nli, entailment_id


pashe_roberta_tok, pashe_roberta, pashe_roberta_entail = pashe_load_nli(
    PASHE_ROBERTA_NLI_MODEL
)
pashe_deberta_tok, pashe_deberta, pashe_deberta_entail = pashe_load_nli(
    PASHE_DEBERTA_NLI_MODEL
)


def pashe_unique_answers(answers):
    normalized = [pashe_normalize(answer) for answer in answers]
    unique = list(dict.fromkeys(normalized))
    return normalized, unique


def pashe_exact_clusters(answers):
    normalized, unique = pashe_unique_answers(answers)
    mapping = {answer: index for index, answer in enumerate(unique)}
    return [mapping[answer] for answer in normalized]


def pashe_embedding_cache(answers, encoder):
    normalized, unique = pashe_unique_answers(answers)
    embeddings = encoder.encode(
        unique,
        normalize_embeddings=True,
        batch_size=PASHE_MODEL_BATCH_SIZE,
    )
    matrix = np.asarray(embeddings) @ np.asarray(embeddings).T
    return normalized, unique, matrix


@torch.inference_mode()
def pashe_nli_cache(answers, tokenizer_nli, model_nli, entailment_id):
    normalized, unique = pashe_unique_answers(answers)
    scores = np.eye(len(unique), dtype=float)
    pairs = [
        (i, j)
        for i in range(len(unique))
        for j in range(len(unique))
        if i != j
    ]
    for start in range(0, len(pairs), PASHE_MODEL_BATCH_SIZE):
        batch = pairs[start:start + PASHE_MODEL_BATCH_SIZE]
        encoded = tokenizer_nli(
            [unique[i] for i, _ in batch],
            [unique[j] for _, j in batch],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)
        probabilities = torch.softmax(
            model_nli(**encoded).logits,
            dim=-1,
        )[:, entailment_id]
        for (i, j), value in zip(batch, probabilities.cpu().tolist()):
            scores[i, j] = value
    return normalized, unique, scores


def pashe_clusters_from_similarity(cache, threshold, bidirectional=False):
    normalized, unique, matrix = cache
    representatives = []
    unique_cluster_ids = []
    for i in range(len(unique)):
        assigned = None
        for cluster_id, representative in enumerate(representatives):
            forward = matrix[i, representative] >= threshold
            backward = matrix[representative, i] >= threshold
            if forward and (backward if bidirectional else True):
                assigned = cluster_id
                break
        if assigned is None:
            assigned = len(representatives)
            representatives.append(i)
        unique_cluster_ids.append(assigned)
    mapping = dict(zip(unique, unique_cluster_ids))
    return [mapping[item] for item in normalized]


_qa_snne_embedding_encoder = None


def qa_snne_get_embedding_encoder():
    global _qa_snne_embedding_encoder
    if _qa_snne_embedding_encoder is None:
        _qa_snne_embedding_encoder = SentenceTransformer(
            QA_SNNE_EMBEDDING_MODEL,
            device=str(device),
        )
    return _qa_snne_embedding_encoder


def qa_snne_embedding_alignment(question, answers):
    encoder = qa_snne_get_embedding_encoder()
    embeddings = np.asarray(encoder.encode(
        [str(question)] + [str(answer) for answer in answers],
        normalize_embeddings=True,
        batch_size=PASHE_MODEL_BATCH_SIZE,
    ))
    return embeddings[1:] @ embeddings[0]



## 4. Weighted risk definitions

For shared semantic clusters \(c\), \(p_k(c)\) is formed by normalising
the raw sampled sequence log-probabilities within condition \(k\) and accumulating
their probability mass by cluster. Let non-negative \(w_k\) sum to one:

\[
\bar p_w(c)=\sum_k w_kp_k(c)
\]

\[
U_{\mathrm{PA-SHE}}^{(w)}
=H(\bar p_w)
\]

### QA-SNNE comparator

For the 20 answers sampled from the **original** image/question, QA-SNNE first
builds a continuous ROUGE-L answer-similarity matrix (S^{text}). SNNE is

\[
U_{SNNE}=-\frac{1}{n}\sum_i\log\sum_{j\ne i}
\exp(S^{text}_{ij}/\tau).
\]

For each alignment variant, question-answer scores \(\alpha_i\) become
\(w_i=\operatorname{softmax}(\beta\alpha_i)\), and bilateral gating gives
\(S^{QA}=\operatorname{diag}(w)S^{text}\operatorname{diag}(w)\). Replacing
\(S^{text}\) with \(S^{QA}\) produces QA-SNNE.

The paper fixes \(n=20\), temperature 1.0, top-k 50, top-p 0.9, \(\beta=10\),
and uses ROUGE-L similarity. It does not report numeric \(\tau,\gamma,\lambda\)
in the paper text, so this notebook declares them transparently as 1.0 and does
not tune them on test. The embedding checkpoint is the PubMed-adapted
`pritamdeka/S-PubMedBert-MS-MARCO`; entailment reuses DeBERTa-large-MNLI;
the cross-encoder is `BAAI/bge-reranker-large`.

In [ ]:
PASHE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]


def pashe_entropy(probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = probabilities[probabilities > 0]
    return float(
        -(probabilities * np.log(probabilities + 1e-12)).sum()
    )


def pashe_distribution(records, cluster_ids, condition, cluster_count):
    indices = [
        i for i, record in enumerate(records)
        if record["condition"] == condition
    ]
    distribution = np.zeros(cluster_count, dtype=float)
    if not indices:
        return distribution

    # Convert generated-answer likelihoods into within-condition sample weights.
    log_probabilities = np.asarray([
        records[i]["sequence_logprob"] for i in indices
    ], dtype=float)
    sample_weights = np.exp(log_probabilities - log_probabilities.max())
    sample_weights /= max(sample_weights.sum(), 1e-12)
    for i, sample_weight in zip(indices, sample_weights):
        distribution[int(cluster_ids[i])] += float(sample_weight)
    return distribution / max(distribution.sum(), 1e-12)


def pashe_weighted_entropy(distributions, condition_weights):
    expected_conditions = set(PASHE_CONDITIONS)
    if set(distributions) != expected_conditions:
        raise ValueError("PA-SHE distributions must contain all four conditions.")
    if set(condition_weights) != expected_conditions:
        raise ValueError("PA-SHE weights must contain all four conditions.")

    weights = np.asarray([
        condition_weights[condition] for condition in PASHE_CONDITIONS
    ], dtype=float)
    if not np.all(np.isfinite(weights)) or np.any(weights < 0):
        raise ValueError("PA-SHE condition weights must be finite and non-negative.")
    if weights.sum() <= 0:
        raise ValueError("At least one PA-SHE condition weight must be positive.")
    weights /= weights.sum()

    first_distribution = np.asarray(
        distributions[PASHE_CONDITIONS[0]], dtype=float
    )
    mixture = np.zeros_like(first_distribution, dtype=float)
    for condition, weight in zip(PASHE_CONDITIONS, weights):
        distribution = np.asarray(distributions[condition], dtype=float)
        if distribution.shape != mixture.shape:
            raise ValueError("PA-SHE condition distributions must share one shape.")
        mass = distribution.sum()
        if mass > 0:
            mixture += float(weight) * (distribution / mass)
    if mixture.sum() <= 0:
        raise ValueError("PA-SHE mixture has no probability mass.")
    return pashe_entropy(mixture / mixture.sum())


def pashe_signals(example, all_cluster_ids, condition_weights):
    records = example["records"]
    record_cluster_ids = all_cluster_ids
    cluster_count = max(all_cluster_ids) + 1

    distributions = {
        condition: pashe_distribution(
            records,
            record_cluster_ids,
            condition,
            cluster_count,
        )
        for condition in PASHE_CONDITIONS
    }
    weighted_pa_she_entropy = pashe_weighted_entropy(
        distributions,
        condition_weights,
    )

    original_distribution = distributions["original"]
    weak_distribution = distributions["weak"]
    distorted_distribution = distributions["distorted"]
    vase = float(jensenshannon(
        weak_distribution + 1e-12,
        distorted_distribution + 1e-12,
        base=2.0,
    ) ** 2)

    signals = {
        "pa_she": weighted_pa_she_entropy,
        "vase": vase,
        "semantic_entropy": pashe_entropy(original_distribution),
        "cluster_count": int(cluster_count),
    }
    return signals


pashe_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def pashe_rouge_l(reference, prediction):
    return float(
        pashe_rouge.score(
            normalize_vqa_metric_text(reference),
            normalize_vqa_metric_text(prediction),
        )["rougeL"].fmeasure
    )


def pashe_best_reference(references, prediction):
    references = [str(reference) for reference in references]
    if not references:
        raise ValueError("At least one accepted reference is required.")
    scores = [pashe_rouge_l(reference, prediction) for reference in references]
    best_index = max(range(len(references)), key=lambda index: (scores[index], -index))
    return references[best_index], float(scores[best_index])


def qa_snne_rouge_similarity_matrix(answers):
    answers = [pashe_normalize(answer) for answer in answers]
    n = len(answers)
    matrix = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            forward = pashe_rouge_l(answers[i], answers[j])
            backward = pashe_rouge_l(answers[j], answers[i])
            matrix[i, j] = matrix[j, i] = 0.5 * (forward + backward)
    return matrix


def qa_snne_score(similarity_matrix, alignment_scores=None):
    """Equations (1)-(4) of Carlini et al.; higher means less certain."""
    similarity = np.asarray(similarity_matrix, dtype=np.float64)
    n = similarity.shape[0]
    if similarity.shape != (n, n) or n < 2:
        raise ValueError("SNNE requires a square matrix with at least two answers.")
    if alignment_scores is not None:
        alignment = np.asarray(alignment_scores, dtype=np.float64)
        if alignment.shape != (n,):
            raise ValueError("QA-SNNE alignment scores must match sampled answers.")
        shifted = QA_SNNE_BETA * alignment
        shifted -= shifted.max()
        relevance = np.exp(shifted)
        relevance /= max(relevance.sum(), 1e-12)
        similarity = np.diag(relevance) @ similarity @ np.diag(relevance)

    row_log_sums = []
    for i in range(n):
        values = np.delete(similarity[i], i) / QA_SNNE_TAU
        maximum = float(values.max())
        row_log_sums.append(
            maximum + np.log(np.exp(values - maximum).sum() + 1e-12)
        )
    return float(-np.mean(row_log_sums))


def qa_snne_signals(example, qa_sample_example):
    answers = [str(answer) for answer in qa_sample_example["answers"]]
    if len(answers) != QA_SNNE_NUM_SAMPLES:
        raise ValueError("QA-SNNE sample count does not match configuration.")
    question = str(example["question"])
    similarity = qa_snne_rouge_similarity_matrix(answers)
    embedding_alignment = qa_snne_embedding_alignment(question, answers)
    return {
        "snne": qa_snne_score(similarity),
        "qa_snne_embedding": qa_snne_score(similarity, embedding_alignment),
        "qa_snne_embedding_alignment_mean": float(np.mean(embedding_alignment)),
    }


## 5. Validation sampling and cache

Only validation examples are sampled here. Test sampling is delayed until the
VQA-Med 2019 clustering has been selected and locked.


In [ ]:
PASHE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]
pashe_datasets = {
    "validation": val_dataset,
    "test": test_dataset,
}


def pashe_dataset_signature(dataset, evaluation_size):
    digest = hashlib.sha256()
    for dataset_index in range(evaluation_size):
        raw = dataset.dataset[dataset_index]
        digest.update(str(dataset_index).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw["question"]).encode("utf-8"))
        digest.update(b"\0")
        digest.update(json.dumps(
            raw.get("answers", [raw["answer"]]), ensure_ascii=False
        ).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def pashe_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if PASHE_MAX_EXAMPLES is None
        else min(int(PASHE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = PASHE_CHECKPOINT_PATH.stat()
    cache_configuration = {
        "cache_schema_version": PASHE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": pashe_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(PASHE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "samples_per_condition": PASHE_NUM_SAMPLES,
        "maximum_new_tokens": PASHE_MAX_NEW_TOKENS,
        "temperature": PASHE_TEMPERATURE,
        "top_p": PASHE_TOP_P,
        "seed": PASHE_RANDOM_SEED,
        "perturbation_version": PASHE_PERTURBATION_VERSION,
        "conditions": PASHE_CONDITIONS,
    }
    cache_hash = hashlib.sha256(
        json.dumps(cache_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = PASHE_CACHE_DIR / f"{split_name}_samples_{cache_hash}.jsonl"

    cached_examples = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as cache_file:
            for line_number, line in enumerate(cache_file, start=1):
                if not line.strip():
                    continue
                try:
                    example = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete cache line {line_number}: {cache_path}")
                    continue
                dataset_index = int(example.get("dataset_index", -1))
                if example.get("cache_schema_version") != PASHE_CACHE_SCHEMA_VERSION:
                    raise ValueError("PA-SHE sample-cache schema mismatch.")
                if example.get("split") != split_name:
                    raise ValueError("PA-SHE sample-cache split mismatch.")
                if not 0 <= dataset_index < evaluation_size:
                    raise ValueError("Cached dataset index is outside this run.")
                condition_counts = {
                    condition: sum(
                        record.get("condition") == condition
                        for record in example.get("records", [])
                    )
                    for condition in PASHE_CONDITIONS
                }
                if any(
                    count != PASHE_NUM_SAMPLES
                    for count in condition_counts.values()
                ):
                    raise ValueError("Cached condition/sample counts do not match.")
                if not example.get("references"):
                    raise ValueError("Cached VQA-Med 2019 references are missing.")
                if dataset_index in cached_examples:
                    raise ValueError("Duplicate dataset index in PA-SHE cache.")
                cached_examples[dataset_index] = example

    pending_indices = [
        index for index in range(evaluation_size)
        if index not in cached_examples
    ]
    print({
        "split": split_name,
        "sample_cache": str(cache_path),
        "cached_examples": len(cached_examples),
        "pending_examples": len(pending_indices),
    })

    split_seed_offset = 0 if split_name == "validation" else 10_000_000
    with cache_path.open("a", encoding="utf-8") as cache_file:
        for dataset_index in tqdm(
            pending_indices,
            desc=f"Frozen-VQA sampling: {split_name}",
        ):
            example_seed = PASHE_RANDOM_SEED + split_seed_offset + dataset_index * 1009
            random.seed(example_seed)
            np.random.seed(example_seed)
            torch.manual_seed(example_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(example_seed)
            example = pashe_collect_example(
                dataset=dataset,
                dataset_index=dataset_index,
                split_name=split_name,
            )
            cache_file.write(json.dumps(example, ensure_ascii=False) + "\n")
            cache_file.flush()
            cached_examples[dataset_index] = example

    return (
        [cached_examples[index] for index in range(evaluation_size)],
        cache_path,
    )





def qa_snne_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if PASHE_MAX_EXAMPLES is None
        else min(int(PASHE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = PASHE_CHECKPOINT_PATH.stat()
    configuration = {
        "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": pashe_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(PASHE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "num_samples": QA_SNNE_NUM_SAMPLES,
        "maximum_new_tokens": PASHE_MAX_NEW_TOKENS,
        "temperature": QA_SNNE_TEMPERATURE,
        "top_k": QA_SNNE_TOP_K,
        "top_p": QA_SNNE_TOP_P,
        "seed": PASHE_RANDOM_SEED,
        "input_condition": "original image and original question",
    }
    cache_hash = hashlib.sha256(
        json.dumps(configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = PASHE_CACHE_DIR / f"{split_name}_qa_snne_samples_{cache_hash}.jsonl"
    cached = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete QA-SNNE cache line {line_number}")
                    continue
                index = int(record.get("dataset_index", -1))
                if record.get("cache_schema_version") != QA_SNNE_CACHE_SCHEMA_VERSION:
                    raise ValueError("QA-SNNE sample-cache schema mismatch.")
                if record.get("split") != split_name or not 0 <= index < evaluation_size:
                    raise ValueError("QA-SNNE sample-cache split/index mismatch.")
                if len(record.get("answers", [])) != QA_SNNE_NUM_SAMPLES:
                    raise ValueError("QA-SNNE cached sample count mismatch.")
                if index in cached:
                    raise ValueError("Duplicate index in QA-SNNE sample cache.")
                cached[index] = record

    pending = [index for index in range(evaluation_size) if index not in cached]
    print({
        "split": split_name,
        "qa_snne_sample_cache": str(cache_path),
        "cached_examples": len(cached),
        "pending_examples": len(pending),
        "samples_per_example": QA_SNNE_NUM_SAMPLES,
    })
    split_offset = 30_000_000 if split_name == "validation" else 40_000_000
    with cache_path.open("a", encoding="utf-8") as handle:
        for index in tqdm(pending, desc=f"QA-SNNE sampling: {split_name}"):
            seed = PASHE_RANDOM_SEED + split_offset + index * 1013
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
            image, question, _ = dataset[index]
            record = {
                "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
                "split": split_name,
                "dataset_index": int(index),
                "question": str(question),
                "answers": qa_snne_generate_samples(
                    image, question, QA_SNNE_NUM_SAMPLES
                ),
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            cached[index] = record
    return [cached[index] for index in range(evaluation_size)], cache_path
# Selection starts with validation only. Test sampling occurs in section 7,
# after PASHE_LOCKED_CLUSTERING has been assigned.
pashe_validation_examples, pashe_validation_sample_cache_path = pashe_collect_split(
    "validation",
    pashe_datasets["validation"],
)
pashe_examples_by_split = {"validation": pashe_validation_examples}
pashe_sample_cache_paths = {"validation": pashe_validation_sample_cache_path}
qa_snne_validation_examples, qa_snne_validation_sample_cache_path = (
    qa_snne_collect_split("validation", pashe_datasets["validation"])
)
qa_snne_examples_by_split = {"validation": qa_snne_validation_examples}
qa_snne_sample_cache_paths = {
    "validation": qa_snne_validation_sample_cache_path
}

print({
    "validation_examples": len(pashe_validation_examples),
    "test_sampled_before_selection": False,
})
display(pd.DataFrame([{
    "split": example["split"],
    "index": example["dataset_index"],
    "question": example["question"],
    "reference": example["reference"],
    "greedy": example["greedy"],
} for example in pashe_validation_examples[:5]]))

## 6. Validation clustering candidates under Equal weighting

Build the predeclared 13-candidate VQA-Med 2019 clustering grid on validation using Equal
condition weights. Alternative condition weights are not involved in
clustering selection and are evaluated later as locked-test sensitivity.


In [ ]:
def pashe_clusters_for_configurations(answers, clustering_configurations):
    clustering_configurations = list(clustering_configurations)
    unsupported = set(clustering_configurations) - set(PASHE_SELECTION_CANDIDATES)
    if unsupported:
        raise ValueError(
            "Unsupported VQA-Med 2019 clustering configuration(s): "
            + ", ".join(sorted(unsupported))
        )
    requested = set(clustering_configurations)
    configurations = []
    if "Exact text" in requested:
        configurations.append(("Exact text", pashe_exact_clusters(answers)))

    embedding_specs = [
        ("SBERT", PASHE_SBERT_THRESHOLDS, pashe_sbert),
        ("BGE", PASHE_BGE_THRESHOLDS, pashe_bge),
    ]
    for method_name, thresholds, encoder in embedding_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = pashe_embedding_cache(answers, encoder)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    pashe_clusters_from_similarity(
                        score_cache, threshold, bidirectional=False
                    ),
                ))

    nli_specs = [
        (
            "RoBERTa-NLI", PASHE_ROBERTA_NLI_THRESHOLDS,
            pashe_roberta_tok, pashe_roberta, pashe_roberta_entail,
        ),
        (
            "DeBERTa-NLI", PASHE_DEBERTA_NLI_THRESHOLDS,
            pashe_deberta_tok, pashe_deberta, pashe_deberta_entail,
        ),
    ]
    for method_name, thresholds, tok, mdl, entail_id in nli_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = pashe_nli_cache(answers, tok, mdl, entail_id)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    pashe_clusters_from_similarity(
                        score_cache, threshold, bidirectional=True
                    ),
                ))
    if {name for name, _ in configurations} != set(clustering_configurations):
        raise ValueError("Failed to construct every requested clustering.")
    return configurations


def pashe_build_weighted_feature_frame(
    split_name,
    examples,
    sample_cache_path,
    clustering_configurations,
    weight_schemes,
):
    clustering_configurations = list(clustering_configurations)
    weight_schemes = list(weight_schemes)
    unknown_weights = set(weight_schemes) - set(PASHE_CONDITION_WEIGHT_SCHEMES)
    if unknown_weights:
        raise ValueError("Unknown condition-weight scheme(s): " + ", ".join(unknown_weights))
    feature_configuration = {
        "feature_schema_version": PASHE_FEATURE_SCHEMA_VERSION,
        "metric_normalization_version": 1,
        "question_router_version": PASHE_QUESTION_ROUTER_VERSION,
        "split": split_name,
        "sample_cache": sample_cache_path.name,
        "clustering_configurations": clustering_configurations,
        "weight_schemes": weight_schemes,
        "condition_weights": {
            name: PASHE_CONDITION_WEIGHT_SCHEMES[name] for name in weight_schemes
        },
        "sequence_weighting": "sequence-probability weighting",
        "sbert_model": PASHE_SBERT_MODEL,
        "sbert_thresholds": PASHE_SBERT_THRESHOLDS,
        "bge_model": PASHE_BGE_MODEL,
        "bge_thresholds": PASHE_BGE_THRESHOLDS,
        "roberta_nli_model": PASHE_ROBERTA_NLI_MODEL,
        "roberta_nli_thresholds": PASHE_ROBERTA_NLI_THRESHOLDS,
        "deberta_nli_model": PASHE_DEBERTA_NLI_MODEL,
        "deberta_nli_thresholds": PASHE_DEBERTA_NLI_THRESHOLDS,
        "qa_snne_sample_cache": qa_snne_sample_cache_paths[split_name].name,
        "qa_snne_num_samples": QA_SNNE_NUM_SAMPLES,
        "qa_snne_beta": QA_SNNE_BETA,
        "qa_snne_tau": QA_SNNE_TAU,
        "qa_snne_embedding_model": QA_SNNE_EMBEDDING_MODEL,
    }
    feature_hash = hashlib.sha256(
        json.dumps(feature_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    feature_path = PASHE_CACHE_DIR / f"{split_name}_weighted_features_{feature_hash}.csv"

    if feature_path.exists():
        frame = pd.read_csv(feature_path, keep_default_na=False)
        expected_rows = (
            len(examples) * len(clustering_configurations) * len(weight_schemes)
        )
        required_columns = {
            "split", "dataset_index", "clustering", "weight_scheme",
            "rougeL", "reference", "reference_options", "prediction",
            "answer_type",
            "predicted_question_type", "pa_she", "semantic_entropy",
            "cluster_count", "snne", "qa_snne_embedding",
        }
        if required_columns - set(frame.columns):
            raise ValueError("Cached weighted VQA-Med 2019 features are incomplete.")
        if len(frame) != expected_rows:
            raise ValueError("Cached weighted VQA-Med 2019 feature row count is incorrect.")
        if set(frame["clustering"]) != set(clustering_configurations):
            raise ValueError("Cached weighted VQA-Med 2019 clustering set is incorrect.")
        if set(frame["weight_scheme"]) != set(weight_schemes):
            raise ValueError("Cached VQA-Med 2019 weight-scheme set is incorrect.")
        print(f"Loaded {split_name} weighted features from {feature_path}")
        return frame, feature_path

    feature_rows = []
    for example in tqdm(examples, desc=f"Weighted semantic features: {split_name}"):
        answers = [record["answer"] for record in example["records"]]
        configurations = pashe_clusters_for_configurations(
            answers, clustering_configurations
        )
        accepted_references = example.get("references", [example["reference"]])
        best_reference, rouge_l = pashe_best_reference(
            accepted_references, example["greedy"]
        )
        normalized_references = {pashe_normalize(value) for value in accepted_references}
        answer_type = (
            "Closed (yes/no)"
            if normalized_references and normalized_references <= {"yes", "no"}
            else "Open-ended"
        )
        qa_sample_example = qa_snne_examples_by_split[split_name][
            int(example["dataset_index"])
        ]
        qa_uncertainty = qa_snne_signals(example, qa_sample_example)
        for clustering, cluster_ids in configurations:
            for weight_scheme in weight_schemes:
                row = {
                    "split": split_name,
                    "dataset_index": int(example["dataset_index"]),
                    "clustering": clustering,
                    "weight_scheme": weight_scheme,
                    "rougeL": rouge_l,
                    "reference": best_reference,
                    "reference_options": " # ".join(accepted_references),
                    "prediction": example["greedy"],
                    "answer_type": answer_type,
                    "predicted_question_type": pashe_predict_question_type(
                        example["question"]
                    ),
                }
                row.update(pashe_signals(
                    example,
                    cluster_ids,
                    PASHE_CONDITION_WEIGHT_SCHEMES[weight_scheme],
                ))
                row.update(qa_uncertainty)
                feature_rows.append(row)

    frame = pd.DataFrame(feature_rows)
    frame.to_csv(feature_path, index=False)
    print(f"Saved {split_name} weighted features to {feature_path}")
    return frame, feature_path


pashe_validation_features, pashe_validation_feature_cache_path = (
    pashe_build_weighted_feature_frame(
        "validation",
        pashe_validation_examples,
        pashe_validation_sample_cache_path,
        PASHE_SELECTION_CANDIDATES,
        list(PASHE_CONDITION_WEIGHT_SCHEMES),
    )
)
display(pashe_validation_features.head())
print({
    "dataset": "VQA-Med 2019",
    "selection_split": "validation",
    "selection_weight_scheme": PASHE_PRIMARY_WEIGHT_SCHEME,
    "selection_candidates": PASHE_SELECTION_CANDIDATES,
    "sequence_weighting": "sequence-probability weighting",
    "test_features_built_before_selection": False,
})


## 7. Validation clustering selection and locked weight sensitivity

Validation PA-SHE AUROC at `ROUGE-L < 0.50` separately selects the predicted-closed and
predicted-open route clusterers; AUPRC and candidate order break ties. Both are locked before
test sampling. Official test evaluates Equal weighting as primary and the
other declared weights as sensitivity, including open-ended safety.

QA-SNNE alignment variant is selected independently by validation AUROC (AUPRC and declared order break ties) and locked before its official-test sample cache is created. Test results never choose the variant.

In [ ]:
PASHE_BASELINE_RISK_COLUMNS = {
    "Vision-Amplified Semantic Entropy": "vase",
    "SE": "semantic_entropy",
    "SNNE": "snne",
    "QA-SNNE · Embedding": "qa_snne_embedding",
}
PASHE_WEIGHTED_RISK_COLUMNS = {"PA-SHE": "pa_she"}
PASHE_RISK_COLUMNS = {**PASHE_BASELINE_RISK_COLUMNS, **PASHE_WEIGHTED_RISK_COLUMNS}


def pashe_safe_metrics(labels, scores):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    if labels.shape != scores.shape:
        raise ValueError("Labels and uncertainty scores must align.")
    if not np.isfinite(scores).all():
        raise ValueError("Uncertainty scores contain NaN or infinity.")
    if np.unique(labels).size < 2:
        return np.nan, np.nan
    return roc_auc_score(labels, scores), average_precision_score(labels, scores)


def pashe_select_clustering(validation_features, selection_subset):
    expected_examples = validation_features["dataset_index"].nunique()
    rows = []
    for candidate_order, clustering in enumerate(PASHE_SELECTION_CANDIDATES):
        group = validation_features[
            validation_features["clustering"] == clustering
        ].sort_values("dataset_index")
        if len(group) != expected_examples:
            raise ValueError(f"Incomplete validation features for {clustering}.")
        failures = (
            group["rougeL"].to_numpy() < PASHE_PRIMARY_LABEL_THRESHOLD
        ).astype(int)
        auroc, auprc = pashe_safe_metrics(failures, group["pa_she"])
        rows.append({
            "selection_split": "validation",
            "selection_subset": selection_subset,
            "selection_weight_scheme": PASHE_PRIMARY_WEIGHT_SCHEME,
            "test_used_for_selection": False,
            "candidate_order": candidate_order,
            "label_threshold": PASHE_PRIMARY_LABEL_THRESHOLD,
            "clustering": clustering,
            "sequence_weighting": "sequence-probability weighting",
            "examples": len(group),
            "failure_prevalence": failures.mean(),
            "validation_AUROC": auroc,
            "validation_AUPRC": auprc,
        })
    selection = pd.DataFrame(rows)
    ranked = selection.sort_values(
        ["validation_AUROC", "validation_AUPRC", "candidate_order"],
        ascending=[False, False, True], kind="mergesort",
    )
    if ranked.empty or pd.isna(ranked.iloc[0]["validation_AUROC"]):
        raise ValueError("Validation labels cannot select clustering.")
    return selection, str(ranked.iloc[0]["clustering"])


closed_validation_features = pashe_validation_features[
    (pashe_validation_features["predicted_question_type"] == "Closed")
    & (pashe_validation_features["weight_scheme"] == PASHE_PRIMARY_WEIGHT_SCHEME)
].copy()
open_validation_features = pashe_validation_features[
    (pashe_validation_features["predicted_question_type"] == "Open")
    & (pashe_validation_features["weight_scheme"] == PASHE_PRIMARY_WEIGHT_SCHEME)
].copy()
if closed_validation_features.empty or open_validation_features.empty:
    raise ValueError("Question router must produce closed and open validation routes.")
pashe_validation_selection, PASHE_LOCKED_CLUSTERING = pashe_select_clustering(
    closed_validation_features, "Predicted closed"
)
pashe_open_validation_selection, PASHE_OPEN_LOCKED_CLUSTERING = pashe_select_clustering(
    open_validation_features, "Predicted open"
)

qa_validation_base = pashe_validation_features[
    (pashe_validation_features["clustering"] == PASHE_SELECTION_CANDIDATES[0])
    & (pashe_validation_features["weight_scheme"] == PASHE_PRIMARY_WEIGHT_SCHEME)
].sort_values("dataset_index")
qa_failures = (
    qa_validation_base["rougeL"].to_numpy() < PASHE_PRIMARY_LABEL_THRESHOLD
).astype(int)
qa_selection_rows = []
for variant_order, (variant, column) in enumerate(QA_SNNE_VARIANTS.items()):
    auroc, auprc = pashe_safe_metrics(qa_failures, qa_validation_base[column])
    qa_selection_rows.append({
        "selection_split": "validation",
        "test_used_for_selection": False,
        "variant_order": variant_order,
        "label_threshold": PASHE_PRIMARY_LABEL_THRESHOLD,
        "variant": variant,
        "column": column,
        "examples": len(qa_validation_base),
        "failure_prevalence": qa_failures.mean(),
        "validation_AUROC": auroc,
        "validation_AUPRC": auprc,
    })
qa_snne_validation_selection = pd.DataFrame(qa_selection_rows)
qa_ranked = qa_snne_validation_selection.sort_values(
    ["validation_AUROC", "validation_AUPRC", "variant_order"],
    ascending=[False, False, True], kind="mergesort",
)
if qa_ranked.empty or pd.isna(qa_ranked.iloc[0]["validation_AUROC"]):
    raise ValueError("Validation labels cannot select a QA-SNNE variant.")
QA_SNNE_LOCKED_VARIANT = str(qa_ranked.iloc[0]["variant"])
QA_SNNE_LOCKED_COLUMN = str(qa_ranked.iloc[0]["column"])
QA_SNNE_LOCKED_METHOD = f"QA-SNNE · {QA_SNNE_LOCKED_VARIANT}"

selection_path = PASHE_CACHE_DIR / "validation_selected_clustering.json"
selection_payload = {
    "dataset": "VQA-Med 2019",
    "selection_split": "validation",
    "selection_weight_scheme": PASHE_PRIMARY_WEIGHT_SCHEME,
    "test_used_for_selection": False,
    "primary_label_definition": "ROUGE-L < 0.50",
    "selection_metric": "AUROC; AUPRC tie-breaker; candidate order final tie-breaker",
    "eligible_candidates": PASHE_SELECTION_CANDIDATES,
    "sequence_weighting": "sequence-probability weighting",
    "closed_route_locked_clustering": PASHE_LOCKED_CLUSTERING,
    "open_route_locked_clustering": PASHE_OPEN_LOCKED_CLUSTERING,
    "qa_snne_locked_variant": QA_SNNE_LOCKED_VARIANT,
    "qa_snne_locked_column": QA_SNNE_LOCKED_COLUMN,
    "qa_snne_candidates": qa_snne_validation_selection.drop(
        columns="variant_order"
    ).to_dict(orient="records"),
    "closed_route_candidates": pashe_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
    "open_route_candidates": pashe_open_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
}
selection_path.write_text(
    json.dumps(selection_payload, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
pashe_validation_selection.to_csv(
    PASHE_CACHE_DIR / "validation_clustering_selection.csv", index=False
)
pashe_open_validation_selection.to_csv(
    PASHE_CACHE_DIR / "validation_open_ended_clustering_selection.csv", index=False
)
qa_snne_validation_selection.to_csv(
    PASHE_CACHE_DIR / "validation_qa_snne_selection.csv", index=False
)

qa_snne_test_examples, qa_snne_test_sample_cache_path = (
    qa_snne_collect_split("test", pashe_datasets["test"])
)
qa_snne_examples_by_split["test"] = qa_snne_test_examples
qa_snne_sample_cache_paths["test"] = qa_snne_test_sample_cache_path
pashe_test_examples, pashe_test_sample_cache_path = pashe_collect_split(
    "test", pashe_datasets["test"]
)
pashe_examples_by_split["test"] = pashe_test_examples
pashe_sample_cache_paths["test"] = pashe_test_sample_cache_path
locked_clusterings = list(dict.fromkeys([
    PASHE_LOCKED_CLUSTERING, PASHE_OPEN_LOCKED_CLUSTERING,
]))
pashe_test_features, pashe_test_feature_cache_path = (
    pashe_build_weighted_feature_frame(
        "test",
        pashe_test_examples,
        pashe_test_sample_cache_path,
        locked_clusterings,
        list(PASHE_CONDITION_WEIGHT_SCHEMES),
    )
)
pashe_locked_test_features = pashe_test_features[
    (pashe_test_features["clustering"] == PASHE_LOCKED_CLUSTERING)
    & (pashe_test_features["predicted_question_type"] == "Closed")
].sort_values(["dataset_index", "weight_scheme"]).copy()
pashe_open_locked_test_features = pashe_test_features[
    (pashe_test_features["clustering"] == PASHE_OPEN_LOCKED_CLUSTERING)
    & (pashe_test_features["predicted_question_type"] == "Open")
].sort_values(["dataset_index", "weight_scheme"]).copy()
expected_routed_rows = (
    pashe_test_features["dataset_index"].nunique()
    * len(PASHE_CONDITION_WEIGHT_SCHEMES)
)
pashe_routed_test_features = pd.concat(
    [pashe_locked_test_features, pashe_open_locked_test_features],
    ignore_index=True,
).sort_values(["dataset_index", "weight_scheme"])
if len(pashe_routed_test_features) != expected_routed_rows:
    raise ValueError("Question-type routing must cover every test scheme once.")


def pashe_validation_percentile(validation_scores, query_scores):
    reference = np.sort(np.asarray(validation_scores, dtype=float))
    query = np.asarray(query_scores, dtype=float)
    if reference.size == 0:
        raise ValueError("Cannot calibrate from an empty validation route.")
    return np.searchsorted(reference, query, side="right") / reference.size


for weight_scheme in PASHE_CONDITION_WEIGHT_SCHEMES:
    closed_validation_locked = pashe_validation_features[
        (pashe_validation_features["predicted_question_type"] == "Closed")
        & (pashe_validation_features["weight_scheme"] == weight_scheme)
        & (pashe_validation_features["clustering"] == PASHE_LOCKED_CLUSTERING)
    ]
    open_validation_locked = pashe_validation_features[
        (pashe_validation_features["predicted_question_type"] == "Open")
        & (pashe_validation_features["weight_scheme"] == weight_scheme)
        & (pashe_validation_features["clustering"] == PASHE_OPEN_LOCKED_CLUSTERING)
    ]
    closed_test_mask = pashe_locked_test_features["weight_scheme"] == weight_scheme
    open_test_mask = pashe_open_locked_test_features["weight_scheme"] == weight_scheme
    for _, risk_column in PASHE_RISK_COLUMNS.items():
        calibrated_column = f"calibrated_{risk_column}"
        pashe_locked_test_features.loc[
            closed_test_mask, calibrated_column
        ] = pashe_validation_percentile(
            closed_validation_locked[risk_column],
            pashe_locked_test_features.loc[closed_test_mask, risk_column],
        )
        pashe_open_locked_test_features.loc[
            open_test_mask, calibrated_column
        ] = pashe_validation_percentile(
            open_validation_locked[risk_column],
            pashe_open_locked_test_features.loc[open_test_mask, risk_column],
        )
pashe_routed_test_features = pd.concat(
    [pashe_locked_test_features, pashe_open_locked_test_features],
    ignore_index=True,
).sort_values(["dataset_index", "weight_scheme"])
pashe_features = pashe_routed_test_features

evaluation_subsets = {
    "All": pashe_routed_test_features,
    "Closed (yes/no)": pashe_routed_test_features[
        pashe_routed_test_features["answer_type"] == "Closed (yes/no)"
    ],
    "Open-ended": pashe_routed_test_features[
        pashe_routed_test_features["answer_type"] == "Open-ended"
    ],
}
pashe_result_rows = []
for evaluation_subset, subset_features in evaluation_subsets.items():
    for label_threshold in PASHE_LABEL_THRESHOLDS:
        for weight_scheme, group in subset_features.groupby("weight_scheme", sort=False):
            failures = (group["rougeL"].to_numpy() < label_threshold).astype(int)
            for method, column in PASHE_RISK_COLUMNS.items():
                auroc, auprc = pashe_safe_metrics(
                    failures, group[f"calibrated_{column}"]
                )
                pashe_result_rows.append({
                    "evaluation_split": "official test",
                    "evaluation_subset": evaluation_subset,
                    "examples": len(group),
                    "label_threshold": label_threshold,
                    "failure_prevalence": failures.mean() if len(failures) else np.nan,
                    "weight_scheme": weight_scheme,
                    "clustering": "Question-type routed",
                    "sequence_weighting": "sequence-probability weighting",
                    "method": method,
                    "AUROC": auroc,
                    "AUPRC": auprc,
                })

pashe_results = pd.DataFrame(pashe_result_rows)
pashe_primary_results = pashe_results[
    (pashe_results["evaluation_subset"] == "All")
    & np.isclose(pashe_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].copy()
primary_weight_results = pashe_primary_results[
    pashe_primary_results["method"].isin(PASHE_WEIGHTED_RISK_COLUMNS)
].sort_values(["method", "AUROC", "AUPRC"], ascending=[True, False, False])
primary_baselines = pashe_primary_results[
    (pashe_primary_results["weight_scheme"] == PASHE_PRIMARY_WEIGHT_SCHEME)
    & pashe_primary_results["method"].isin(PASHE_BASELINE_RISK_COLUMNS)
].sort_values(["AUROC", "AUPRC"], ascending=False)
pashe_label_sensitivity = pashe_results[
    (pashe_results["evaluation_subset"] == "All")
    & (pashe_results["method"] == "PA-SHE")
].sort_values(["weight_scheme", "label_threshold"])
pashe_open_ended_results = pashe_results[
    pashe_results["evaluation_subset"] == "Open-ended"
].copy()
pashe_open_ended_primary_results = pashe_open_ended_results[
    np.isclose(pashe_open_ended_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].sort_values(["AUROC", "AUPRC"], ascending=False)

pashe_results.to_csv(PASHE_CACHE_DIR / "locked_test_weighted_safety.csv", index=False)
pashe_open_ended_results.to_csv(
    PASHE_CACHE_DIR / "locked_test_weighted_open_ended_safety.csv", index=False
)
print("VALIDATION-ONLY QA-SNNE variant selection:")
display(qa_snne_validation_selection.drop(columns="variant_order").round(4))
print("Locked QA-SNNE variant before test sampling:", QA_SNNE_LOCKED_VARIANT)
print("VALIDATION-ONLY VQA-Med 2019 clustering selection under Equal weighting:")
display(pashe_validation_selection.drop(columns="candidate_order").round(4))
print("VALIDATION-ONLY open-ended clustering selection:")
display(pashe_open_validation_selection.drop(columns="candidate_order").round(4))
print("Closed-route locked clustering:", PASHE_LOCKED_CLUSTERING)
print("Open-route locked clustering:", PASHE_OPEN_LOCKED_CLUSTERING)
print("OFFICIAL TEST weight sensitivity at ROUGE-L < 0.50")
display(primary_weight_results.round(4))
print("OFFICIAL TEST baselines under primary Equal weighting")
display(primary_baselines.round(4))
print("OFFICIAL TEST open-ended primary safety")
display(pashe_open_ended_primary_results.round(4))
print("Saved validation selection protocol:", selection_path)


## 8. Validation selection and locked test weight sensitivity

The clustering-selection chart is validation-only. Weight and label plots
use official test with the selected clustering held fixed.


In [ ]:
# Validation-only overall and open-ended clustering selections.
validation_plots = [
    ("Predicted-closed validation", pashe_validation_selection),
    ("Predicted-open validation", pashe_open_validation_selection),
]
fig, axes = plt.subplots(2, 2, figsize=(20, 10))
for row_index, (subset_label, selection_frame) in enumerate(validation_plots):
    validation_plot = selection_frame.sort_values("candidate_order")
    axes[row_index, 0].bar(
        validation_plot["clustering"], validation_plot["validation_AUROC"]
    )
    axes[row_index, 1].bar(
        validation_plot["clustering"],
        validation_plot["validation_AUPRC"],
        color="#f58518",
    )
    axes[row_index, 0].set_title(f"{subset_label}: PA-SHE AUROC")
    axes[row_index, 1].set_title(f"{subset_label}: PA-SHE AUPRC")
    for axis in axes[row_index]:
        axis.set_ylim(0, 1)
        axis.tick_params(axis="x", rotation=45)
        axis.grid(axis="y", alpha=0.25)
fig.suptitle("VQA-Med 2019 validation locks under Equal condition weighting")
plt.tight_layout()
plt.show()

# Overall test weight sensitivity uses the overall validation lock.
weight_heatmap = pashe_primary_results[
    pashe_primary_results["method"] == "PA-SHE"
].pivot(index="weight_scheme", columns="method", values="AUROC")
plt.figure(figsize=(10, max(5, 0.55 * len(weight_heatmap))))
sns.heatmap(weight_heatmap, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="viridis")
plt.title("Official-test AUROC with question-type routing")
plt.xlabel("Uncertainty method")
plt.ylabel("Condition-weight scheme")
plt.tight_layout()
plt.show()

label_sensitivity = pashe_label_sensitivity.pivot(
    index="weight_scheme",
    columns="label_threshold",
    values="AUROC",
)
print("PA-SHE test label sensitivity; clustering remains locked")
display(label_sensitivity.round(4))


# QA-SNNE variant choice is validation-only; the official-test values below
# are reporting, never a second selection step.
qa_validation_plot = qa_snne_validation_selection.sort_values("variant_order")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(qa_validation_plot["variant"], qa_validation_plot["validation_AUROC"])
axes[1].bar(
    qa_validation_plot["variant"],
    qa_validation_plot["validation_AUPRC"],
    color="#f58518",
)
axes[0].set_title("Validation QA-SNNE AUROC")
axes[1].set_title("Validation QA-SNNE AUPRC")
for axis in axes:
    axis.set_ylim(0, 1)
    axis.tick_params(axis="x", rotation=15)
    axis.grid(axis="y", alpha=0.25)
fig.suptitle(f"Locked QA-SNNE variant: {QA_SNNE_LOCKED_VARIANT}")
plt.tight_layout()
plt.show()

qa_test_primary = pashe_primary_results[
    (pashe_primary_results["weight_scheme"] == PASHE_PRIMARY_WEIGHT_SCHEME)
    & pashe_primary_results["method"].isin(
        ["SNNE"] + [f"QA-SNNE · {variant}" for variant in QA_SNNE_VARIANTS]
    )
].sort_values("AUROC", ascending=False)
print("Official-test SNNE/QA-SNNE comparison; variant already locked on validation")
display(qa_test_primary.round(4))

## 9. Locked selective prediction and qualitative weight audit

Every overall curve and audit row uses the overall validation-locked clustering.
Alternative weights remain sensitivity analyses rather than test-selected settings.


In [ ]:
primary_features = pashe_routed_test_features.copy()
rejection_fractions = np.linspace(0, 0.50, 11)
curve_rows = []
risk_sources = [
    ("Original-condition SE", PASHE_PRIMARY_WEIGHT_SCHEME, "semantic_entropy"),
    ("SNNE", PASHE_PRIMARY_WEIGHT_SCHEME, "snne"),
    (
        f"QA-SNNE · {QA_SNNE_LOCKED_VARIANT} (validation-selected)",
        PASHE_PRIMARY_WEIGHT_SCHEME,
        QA_SNNE_LOCKED_COLUMN,
    ),
]
risk_sources.extend([
    (f"PA-SHE · {scheme}", scheme, "pa_she")
    for scheme in PASHE_CONDITION_WEIGHT_SCHEMES
])

for method_label, weight_scheme, column in risk_sources:
    scheme_features = primary_features[
        primary_features["weight_scheme"] == weight_scheme
    ]
    ordered = scheme_features.sort_values(
        f"calibrated_{column}", ascending=True
    )
    for fraction in rejection_fractions:
        retained_count = max(1, int(round(len(ordered) * (1.0 - fraction))))
        retained = ordered.iloc[:retained_count]
        curve_rows.append({
            "method": method_label,
            "rejected_fraction": fraction,
            "retained_ROUGE-L": retained["rougeL"].mean(),
        })

pashe_rejection = pd.DataFrame(curve_rows)
plt.figure(figsize=(12, 7))
for method, group in pashe_rejection.groupby("method", sort=False):
    plt.plot(
        group["rejected_fraction"],
        group["retained_ROUGE-L"],
        marker="o",
        label=method,
    )
plt.xlabel("Fraction rejected as high risk")
plt.ylabel("Mean ROUGE-L among retained answers")
plt.title("Official test weight sensitivity: question-type routed")
plt.grid(alpha=0.25)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

weight_pivot = primary_features.pivot(
    index="dataset_index",
    columns="weight_scheme",
    values="pa_she",
)
weight_pivot["original_heavy_minus_equal"] = (
    weight_pivot["Original-heavy"] - weight_pivot["Equal"]
)
audit_metadata = primary_features[
    primary_features["weight_scheme"] == PASHE_PRIMARY_WEIGHT_SCHEME
][[
    "dataset_index", "answer_type", "reference", "prediction", "rougeL",
]]
weight_audit = audit_metadata.merge(
    weight_pivot.reset_index(), on="dataset_index", how="left"
)
audit_columns = [
    "dataset_index", "answer_type", "reference", "prediction", "rougeL",
    "Equal", "Original-heavy", "original_heavy_minus_equal",
]
print("Largest increases from Equal to Original-heavy weighting")
display(weight_audit.nlargest(10, "original_heavy_minus_equal")[audit_columns].round(4))
print("Largest decreases from Equal to Original-heavy weighting")
display(weight_audit.nsmallest(10, "original_heavy_minus_equal")[audit_columns].round(4))

## 10. Reporting checklist

- Cite QA-SNNE as Carlini et al., arXiv:2511.01458.
- Report its fixed sampling configuration: 20 original-input samples,
  temperature 1.0, top-k 50, top-p 0.9, beta 10, and ROUGE-L similarity.
- State that tau, gamma, and lambda are fixed to 1.0 in this implementation
  because numeric values are not specified in the paper text.
- Select both PA-SHE route clusterers and the QA-SNNE alignment variant using validation
  AUROC only; lock them before official-test sampling.
- Treat Equal as the primary condition weighting and other weights as declared
  sensitivity analyses.
- Report overall, closed-ended, and open-ended official-test AUROC/AUPRC.
- Do not choose a weight, clustering, or QA-SNNE variant from the test table.

## 11. Main PA-SHE comparison table

The table uses separate predicted-closed and predicted-open VQA-Med 2019 clusterers selected on
validation under Equal weighting and locked before test. Their routed test scores are combined for the overall column. Utility repeats because all uncertainty
scores evaluate the same predictions. Overall, closed-ended, and open-ended safety use the
primary failure definition `ROUGE-L < 0.50`.


In [ ]:
# Final table using separately validation-locked overall/open clustering.
import evaluate
from IPython.display import display

required_names = [
    "pashe_results", "pashe_open_ended_results", "pashe_routed_test_features",
    "PASHE_LOCKED_CLUSTERING", "PASHE_OPEN_LOCKED_CLUSTERING",
    "PASHE_PRIMARY_LABEL_THRESHOLD",
    "QA_SNNE_LOCKED_VARIANT", "QA_SNNE_LOCKED_METHOD",
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError("Run PA-SHE sections 5-7 first. Missing: " + ", ".join(missing_names))

utility_source = pashe_routed_test_features[
    pashe_routed_test_features["weight_scheme"] == PASHE_PRIMARY_WEIGHT_SCHEME
].sort_values("dataset_index").drop_duplicates("dataset_index")
references = [
    normalize_vqa_metric_text(text)
    for text in utility_source["reference"].astype(str)
]
predictions = [
    normalize_vqa_metric_text(text)
    for text in utility_source["prediction"].astype(str)
]
utility = {
    "BLEU-1": 100.0 * float(evaluate.load("bleu").compute(
        predictions=predictions, references=references, max_order=1
    )["bleu"]),
    "ROUGE-L": 100.0 * float(evaluate.load("rouge").compute(
        predictions=predictions, references=references
    )["rougeL"]),
    "METEOR": 100.0 * float(evaluate.load("meteor").compute(
        predictions=predictions, references=references
    )["meteor"]),
}
overall_safety = pashe_results[
    (pashe_results["evaluation_subset"] == "All")
    & np.isclose(pashe_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
]
closed_safety = pashe_results[
    (pashe_results["evaluation_subset"] == "Closed (yes/no)")
    & np.isclose(pashe_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
]
open_safety = pashe_open_ended_results[
    np.isclose(pashe_open_ended_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
]


def lookup(frame, method, weight_scheme):
    match = frame[
        (frame["method"] == method)
        & (frame["weight_scheme"] == weight_scheme)
    ]
    if len(match) != 1:
        raise ValueError(
            f"Expected one locked result for {method}/{weight_scheme}; found {len(match)}"
        )
    row = match.iloc[0]
    return 100.0 * float(row["AUROC"]), 100.0 * float(row["AUPRC"])


specification = [
    ("Semantic entropy", "Original condition", "SE", "Equal"),
    (
        "Semantic nearest-neighbour entropy",
        f"SNNE · ROUGE-L · n={QA_SNNE_NUM_SAMPLES}",
        "SNNE",
        "Equal",
    ),
    ("Vision-Amplified Semantic Entropy", "Vision-Amplified Semantic Entropy", "Vision-Amplified Semantic Entropy", "Equal"),
]
for variant in QA_SNNE_VARIANTS:
    selected_suffix = " · validation-selected" if variant == QA_SNNE_LOCKED_VARIANT else ""
    specification.append((
        "Question-aligned SNNE",
        f"{variant}{selected_suffix} · beta={QA_SNNE_BETA:g}",
        f"QA-SNNE · {variant}",
        "Equal",
    ))
for scheme, weights in PASHE_CONDITION_WEIGHT_SCHEMES.items():
    label = (
        f"{scheme}: O={weights['original']:.2f}, W={weights['weak']:.2f}, "
        f"D={weights['distorted']:.2f}, P={weights['paraphrase']:.2f}"
    )
    specification.append(("Weighted PA-SHE", label, "PA-SHE", scheme))
rows = []
for family, variant, method, scheme in specification:
    overall_auroc, overall_auprc = lookup(overall_safety, method, scheme)
    closed_auroc, closed_auprc = lookup(closed_safety, method, scheme)
    open_auroc, open_auprc = lookup(open_safety, method, scheme)
    rows.append({
        "Uncertainty method": family,
        "Variant / weights": variant,
        ("Utility", "BLEU-1"): utility["BLEU-1"],
        ("Utility", "ROUGE-L"): utility["ROUGE-L"],
        ("Utility", "METEOR"): utility["METEOR"],
        ("Overall safety", "AUROC"): overall_auroc,
        ("Overall safety", "AUPRC"): overall_auprc,
        ("Closed-ended safety", "AUROC"): closed_auroc,
        ("Closed-ended safety", "AUPRC"): closed_auprc,
        ("Open-ended safety", "AUROC"): open_auroc,
        ("Open-ended safety", "AUPRC"): open_auprc,
    })
comparison_df = pd.DataFrame(rows).set_index([
    "Uncertainty method", "Variant / weights"
])
comparison_df.columns = pd.MultiIndex.from_tuples(
    comparison_df.columns, names=["Evaluation dimension", "Metric"]
)
safety_columns = [
    ("Overall safety", "AUROC"), ("Overall safety", "AUPRC"),
    ("Closed-ended safety", "AUROC"), ("Closed-ended safety", "AUPRC"),
    ("Open-ended safety", "AUROC"), ("Open-ended safety", "AUPRC"),
]
maxima = {column: comparison_df[column].max() for column in safety_columns}


def highlight(value, column):
    if pd.notna(value) and np.isclose(value, maxima.get(column, np.nan)):
        return "font-weight: 700; background-color: #e8f1fb;"
    return ""


comparison_styler = (
    comparison_df.style
    .format("{:.2f}", na_rep="—")
    .apply(lambda series: [highlight(value, series.name) for value in series], axis=0)
    .set_caption(
        "VQA-Med 2019 Weighted PA-SHE and QA-SNNE: sequence-probability weighting; "
        f"Closed route={PASHE_LOCKED_CLUSTERING}; "
        f"Open route={PASHE_OPEN_LOCKED_CLUSTERING}"
    )
    .set_table_styles([
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("font-size", "18px"),
            ("font-weight", "700"), ("text-align", "left"),
        ]},
        {"selector": "th", "props": [
            ("background-color", "#f5f5f5"), ("border", "1px solid #aaa"),
            ("padding", "8px"), ("text-align", "center"),
        ]},
        {"selector": "td", "props": [
            ("border", "1px solid #b5b5b5"), ("padding", "8px"),
            ("text-align", "center"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"), ("font-size", "13px"),
            ("width", "100%"),
        ]},
    ])
)
display(comparison_styler)
comparison_df.to_csv("VQAMed2019_PA_SHE_comparison.csv")
with open(
    "VQAMed2019_PA_SHE_comparison.html",
    "w",
    encoding="utf-8",
) as comparison_file:
    comparison_file.write(comparison_styler.to_html())
print("Closed-route locked clustering:", PASHE_LOCKED_CLUSTERING)
print("Open-route locked clustering:", PASHE_OPEN_LOCKED_CLUSTERING)
print("Saved VQAMed2019_PA_SHE_comparison.csv/html")


### Reading the table

- Predicted-closed and predicted-open clustering routes are separately validation-selected, locked, and then combined for overall test evaluation.
- Equal weighting is primary; other condition weights are sensitivity analyses.
- Overall safety uses every test question; closed-ended safety uses accepted references containing only normalized `yes`/`no`; open-ended safety uses all remaining references.
- Official `#` answer alternatives and optional parenthetical text are expanded as accepted references; the highest ROUGE-L match to the shared greedy prediction defines the per-example reference score.
- AUROC and AUPRC detect answers with `ROUGE-L < 0.50`.
- Bold values compare displayed rankings but do not select a new test configuration.
- SNNE and QA-SNNE use their separate original-input sample cache; they do not reuse visually perturbed PA-SHE outputs.
- The QA-SNNE row marked `validation-selected` is the primary paper-method comparison; the other two predeclared variants are reported as sensitivity.